# ML + DL ensembles & blending — final Colab version

Единый ансамблевый ноутбук для **tuned ML + tuned DL** моделей.

Что делает:
- загружает tuned DL-модели из `tuned_summary.csv`
- загружает tuned ML-модели из `final_summary.json` / `final_summary.csv`
- восстанавливает обе группы моделей по их финальным параметрам
- кэширует предсказания на Drive / локально
- проводит единый exhaustive-style ensemble search по объединённому пулу ML+DL моделей
- выбирает победителя по `selection_MAE` на внутренней blend-selection части validation
- считает внешние метрики на `typical holdout` и `full holdout`


In [ ]:
import importlib.util, subprocess, sys, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

REQUIRED_PACKAGES = ["optuna"]
OPTIONAL_PACKAGES = ["xgboost"]
TABPFN_HELPER_PACKAGES = ["huggingface_hub"]

INSTALL_TABPFN = True
RUN_TABPFN_DEFAULT = True

# ВАЖНО:
# 1) Для ensemble-ноутбука используем Hugging Face token, а не TABPFN_TOKEN от browser/license flow.
# 2) Явно целимся в TabPFN-2.5, потому что дефолтный TabPFN-2.6 может уводить в browser login в headless Colab.
TABPFN_ENABLE_HF_LOGIN = True
TABPFN_HF_TOKEN = ""   # не храним токен в ноутбуке; используй Colab Secrets / env vars
TABPFN_FORCE_MODEL_VERSION = "V2_5"

def _has_module(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is not None

def _pip_install(pkg_name: str):
    print(f"Installing {pkg_name} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg_name])

for pkg in REQUIRED_PACKAGES:
    if not _has_module(pkg):
        _pip_install(pkg)

HAS_XGBOOST = _has_module("xgboost")
if not HAS_XGBOOST:
    try:
        _pip_install("xgboost")
        HAS_XGBOOST = _has_module("xgboost")
    except Exception as e:
        HAS_XGBOOST = False
        print(f"xgboost install skipped/failed: {e}")

if INSTALL_TABPFN:
    if not _has_module("tabpfn"):
        _pip_install("tabpfn")
    for helper_pkg in TABPFN_HELPER_PACKAGES:
        if not _has_module(helper_pkg):
            _pip_install(helper_pkg)

TABPFN_HF_LOGIN_STATUS = "not_attempted"

def _read_colab_secret(secret_names):
    if not IN_COLAB:
        return None
    try:
        from google.colab import userdata
    except Exception:
        return None
    for name in secret_names:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    return None

# Только Hugging Face token names. Не используем TABPFN_TOKEN, чтобы не путать HF flow и browser/license flow.
TABPFN_HF_TOKEN_EFFECTIVE = (
    (TABPFN_HF_TOKEN or "").strip()
    or os.environ.get("HF_TOKEN", "").strip()
    or os.environ.get("HUGGING_FACE_HUB_TOKEN", "").strip()
    or os.environ.get("HUGGINGFACEHUB_API_TOKEN", "").strip()
    or os.environ.get("HUGGINGFACE_TOKEN", "").strip()
    or _read_colab_secret(["HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HUGGINGFACEHUB_API_TOKEN", "HUGGINGFACE_TOKEN"])
)

if TABPFN_HF_TOKEN_EFFECTIVE:
    # Дублируем токен в стандартные HF env vars до import tabpfn.
    os.environ["HF_TOKEN"] = TABPFN_HF_TOKEN_EFFECTIVE
    os.environ["HUGGING_FACE_HUB_TOKEN"] = TABPFN_HF_TOKEN_EFFECTIVE
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = TABPFN_HF_TOKEN_EFFECTIVE
    os.environ["HUGGINGFACE_TOKEN"] = TABPFN_HF_TOKEN_EFFECTIVE

if INSTALL_TABPFN and TABPFN_ENABLE_HF_LOGIN:
    try:
        from huggingface_hub import login
        if TABPFN_HF_TOKEN_EFFECTIVE:
            login(token=TABPFN_HF_TOKEN_EFFECTIVE, add_to_git_credential=False, skip_if_logged_in=False)
            prefix = TABPFN_HF_TOKEN_EFFECTIVE[:4]
            suffix = TABPFN_HF_TOKEN_EFFECTIVE[-4:]
            TABPFN_HF_LOGIN_STATUS = f"ok:{prefix}...{suffix}"
        else:
            TABPFN_HF_LOGIN_STATUS = "missing_token"
    except Exception as e:
        TABPFN_HF_LOGIN_STATUS = f"failed:{type(e).__name__}:{e}"

print("TabPFN HF login status:", TABPFN_HF_LOGIN_STATUS)
print("TabPFN target model version:", TABPFN_FORCE_MODEL_VERSION)
print("If TabPFN-2.5 is used for the first time, make sure the license is accepted on Hugging Face: https://huggingface.co/Prior-Labs/tabpfn_2_5")

In [ ]:
from pathlib import Path
import os
import json
import ast
import math
import gc
import random
import sys
import shutil
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

SEED = 2
DURATION_CAP = 960
TARGET_COL = "duration_hours"

# =========================
# Google Drive / Colab I/O
# =========================
IN_COLAB = "google.colab" in sys.modules
USE_GOOGLE_DRIVE = True            # отключи, если не хочешь сохранять на Drive
MOUNT_GOOGLE_DRIVE = True          # в Colab обычно оставляем True
DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE_PROJECT_DIR = DRIVE_ROOT / "diploma_ml_dl_runs"
DRIVE_DATA_DIR = DRIVE_PROJECT_DIR / "data"
DRIVE_ARTIFACTS_DIR = DRIVE_PROJECT_DIR / "artifacts_ml_dl_ensembles_final"
DRIVE_BASELINE_DIR = DRIVE_PROJECT_DIR / "artifacts_dl_baseline_final"

if IN_COLAB and USE_GOOGLE_DRIVE and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    print(f"Google Drive mounted: {DRIVE_ROOT}")
    DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)
    DRIVE_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
else:
    print("Google Drive saving disabled or not running in Colab.")

# Shared cache for TabPFN model weights/checkpoints on Google Drive
TABPFN_CACHE_DIR = DRIVE_PROJECT_DIR / "tabpfn_model_cache"
TABPFN_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("TABPFN_MODEL_CACHE_DIR", str(TABPFN_CACHE_DIR))
print("TABPFN_MODEL_CACHE_DIR =", os.environ.get("TABPFN_MODEL_CACHE_DIR"))

def seed_everything(seed: int = 2):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        if hasattr(torch.backends, "cudnn"):
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    except Exception:
        pass

seed_everything(SEED)


# =========================
# Ensemble experiment config
# =========================
RUN_NAME = datetime.now().strftime("run_%Y%m%d_%H%M%S")
BLEND_FIT_FRAC = 0.50           # первая половина inner-val для fit весов/stacker, вторая — для выбора ансамбля
FORCE_REBUILD_BASE_PREDICTIONS = False
RESUME_IF_PREDICTIONS_EXIST = True

# Модельный пул DL
USE_ALL_TUNED_MODELS = True
MANUAL_MODEL_POOL = []          # если USE_ALL_TUNED_MODELS=False, сюда можно передать список DL-моделей

# Насколько глубоко идти по комбинациям
RUN_ALL_PAIRS = True
RUN_ALL_TRIPLES = True
RUN_TOPK_PREFIX_ENSEMBLES = True
RUN_GREEDY_SEARCH = True
RUN_STACKERS = True
RUN_EXHAUSTIVE_TOPN_SUBSETS = True
EXHAUSTIVE_TOP_N = 14           # exhaustive subsets только для top-N по blend-fit MAE
REFIT_TOP_ENSEMBLES = 250       # сколько лучших по selection-MAE ансамблей пересчитать на holdout
PAIR_WEIGHT_GRID = np.linspace(0.0, 1.0, 101)

# Пути к предыдущим DL-артефактам
CUSTOM_TUNED_SUMMARY_PATH = None
TUNED_SUMMARY_CANDIDATES = [
    Path(CUSTOM_TUNED_SUMMARY_PATH) if CUSTOM_TUNED_SUMMARY_PATH else None,
    Path("./tuned_summary.csv"),
    Path("/content/tuned_summary.csv"),
    Path("/mnt/data/tuned_summary.csv"),
]
if IN_COLAB and USE_GOOGLE_DRIVE:
    TUNED_SUMMARY_CANDIDATES.extend([
        DRIVE_PROJECT_DIR / "artifacts_dl_tuning_final" / "tuned_summary.csv",
        DRIVE_PROJECT_DIR / "artifacts_dl_tuning_final" / "run_latest" / "tuned_summary.csv",
    ])
TUNED_SUMMARY_CANDIDATES = [p for p in TUNED_SUMMARY_CANDIDATES if p is not None]
TUNED_SUMMARY_PATH = next((p for p in TUNED_SUMMARY_CANDIDATES if p.exists()), None)
if TUNED_SUMMARY_PATH is None:
    raise FileNotFoundError(
        "Не найден tuned_summary.csv. Сначала прогони DL_tuning ноутбук и сохрани артефакты на Drive, "
        "или положи tuned_summary.csv рядом с ноутбуком."
    )

tuned_df = pd.read_csv(TUNED_SUMMARY_PATH)
print(f"Loaded tuned summary from: {TUNED_SUMMARY_PATH}")
print(f"Tuned rows: {len(tuned_df)}")

CUSTOM_DATA_PATH = None  # при необходимости укажи сюда явный путь строкой
DATA_CANDIDATES = [
    Path(CUSTOM_DATA_PATH) if CUSTOM_DATA_PATH else None,
    Path("./data_finall_without_TTM.csv"),
    Path("/content/data_finall_without_TTM.csv"),
    Path("/mnt/data/data_finall_without_TTM.csv"),
    Path("./data_finall.csv"),
    Path("/content/data_finall.csv"),
    Path("/mnt/data/data_finall.csv"),
    Path("./new_normal_dataset.csv"),
    Path("/content/new_normal_dataset.csv"),
    Path("/mnt/data/new_normal_dataset.csv"),
]

if IN_COLAB and USE_GOOGLE_DRIVE:
    DATA_CANDIDATES.extend([
        DRIVE_DATA_DIR / "data_finall_without_TTM.csv",
        DRIVE_PROJECT_DIR / "data_finall_without_TTM.csv",
        DRIVE_ROOT / "data_finall_without_TTM.csv",
        DRIVE_DATA_DIR / "data_finall.csv",
        DRIVE_PROJECT_DIR / "data_finall.csv",
        DRIVE_ROOT / "data_finall.csv",
    ])

DATA_CANDIDATES = [p for p in DATA_CANDIDATES if p is not None]

DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Не найден датасет. Положи рядом с ноутбуком data_finall_without_TTM.csv, "
        "или в Google Drive/MyDrive/diploma_dl_runs/data/, "
        "или укажи путь в CUSTOM_DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset from: {DATA_PATH}")

if TARGET_COL not in df.columns:
    raise ValueError(f"В датасете нет целевой колонки {TARGET_COL!r}")

split = int(len(df) * 0.8)
train_full = df.iloc[:split].copy()
test_full = df.iloc[split:].copy()

train_typical = train_full[train_full[TARGET_COL] < DURATION_CAP].copy()
test_typical = test_full[test_full[TARGET_COL] < DURATION_CAP].copy()

Xtrain = train_typical.drop(columns=[TARGET_COL], errors="ignore").copy()
ytrain = train_typical[TARGET_COL].copy()

Xtest_typical = test_typical.drop(columns=[TARGET_COL], errors="ignore").copy()
ytest_typical = test_typical[TARGET_COL].copy()

Xtest_full = test_full.drop(columns=[TARGET_COL], errors="ignore").copy()
ytest_full = test_full[TARGET_COL].copy()

# bool -> 0/1 для torch / sklearn scalers
bool_cols = Xtrain.select_dtypes(include=["bool"]).columns.tolist()
for _frame in (Xtrain, Xtest_typical, Xtest_full):
    if bool_cols:
        _frame[bool_cols] = _frame[bool_cols].astype("int8")

print(f"Rows total: {len(df)} | train_full: {len(train_full)} | train_typical: {len(train_typical)}")
print(f"test_typical: {len(test_typical)} | test_full: {len(test_full)}")


# aliases для совместимости с исходными DL-хелперами
Xtest = Xtest_typical.copy()
ytest = ytest_typical.copy()
seed = SEED


# =========================
# Artifacts / cache dirs
# =========================
LOCAL_ART_DIR = Path("./artifacts_ml_dl_ensembles_final")
LOCAL_ART_DIR.mkdir(parents=True, exist_ok=True)

if IN_COLAB and USE_GOOGLE_DRIVE:
    ART_DIR = DRIVE_ARTIFACTS_DIR
else:
    ART_DIR = LOCAL_ART_DIR
ART_DIR.mkdir(parents=True, exist_ok=True)

RUN_DIR = ART_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

PRED_CACHE_DIR = ART_DIR / "pred_cache"
for _sub in [
    "split_val",
    "split_test_typical",
    "split_test_full",
    "fullfit_test_typical",
    "fullfit_test_full",
]:
    (PRED_CACHE_DIR / _sub).mkdir(parents=True, exist_ok=True)

print(f"Ensemble artifacts root: {ART_DIR}")
print(f"Current run dir: {RUN_DIR}")


In [ ]:
import importlib.util, subprocess, sys

if importlib.util.find_spec('tabpfn') is None:
    print('Installing tabpfn ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'tabpfn'])
else:
    print('tabpfn already installed')


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time, math, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except Exception:
    class _DummyTrialPruned(Exception):
        pass
    class _DummyLogging:
        WARNING = None
        @staticmethod
        def set_verbosity(*args, **kwargs):
            return None
    class _DummyOptuna:
        TrialPruned = _DummyTrialPruned
        logging = _DummyLogging()
    optuna = _DummyOptuna()

try:
    from tabpfn import TabPFNRegressor
    try:
        from tabpfn.constants import ModelVersion
        TABPFN_MODEL_VERSION_ENUM = getattr(ModelVersion, globals().get("TABPFN_FORCE_MODEL_VERSION", "V2_5"), None)
    except Exception:
        ModelVersion = None
        TABPFN_MODEL_VERSION_ENUM = None
    HAS_TABPFN = True
    print("TabPFN доступен ✓")
    print(f"TabPFN HF login status: {globals().get('TABPFN_HF_LOGIN_STATUS', 'unknown')}")
    print(f"TabPFN selected version enum: {TABPFN_MODEL_VERSION_ENUM}")
except ImportError:
    HAS_TABPFN = False
    ModelVersion = None
    TABPFN_MODEL_VERSION_ENUM = None
    print("TabPFN не установлен — pip install tabpfn")

torch.manual_seed(seed)
np.random.seed(seed)

def make_tabpfn_regressor(device: str = "cuda", n_estimators: int = 8):
    if not globals().get("HAS_TABPFN", False):
        raise RuntimeError("TabPFN package is not available.")
    # Для headless Colab принудительно используем Hugging Face-compatible TabPFN-2.5.
    if globals().get("TABPFN_MODEL_VERSION_ENUM", None) is None:
        raise RuntimeError(
            "Current tabpfn build does not expose ModelVersion.V2_5. "
            "Upgrade tabpfn or restart after a clean install."
        )
    reg = TabPFNRegressor.create_default_for_version(globals()["TABPFN_MODEL_VERSION_ENUM"])
    try:
        reg.set_params(device=device, n_estimators=int(n_estimators))
    except Exception:
        # sklearn-style set_params может быть недоступен на некоторых версиях
        try:
            reg.device = device
        except Exception:
            pass
        try:
            reg.n_estimators = int(n_estimators)
        except Exception:
            pass
    return reg


# ═══════════════════════════════════════════════════════════════
#  1. Подготовка данных
# ═══════════════════════════════════════════════════════════════

val_cut = int(len(Xtrain) * 0.8)

X_dl_tr = Xtrain.iloc[:val_cut].values.astype(np.float32)
y_dl_tr = ytrain.iloc[:val_cut].values.astype(np.float32)
X_dl_va = Xtrain.iloc[val_cut:].values.astype(np.float32)
y_dl_va = ytrain.iloc[val_cut:].values.astype(np.float32)

sc = StandardScaler()
X_dl_tr = sc.fit_transform(X_dl_tr)
X_dl_va = sc.transform(X_dl_va)
X_dl_te = sc.transform(Xtest.values.astype(np.float32))

sc_full = StandardScaler()
X_full_s = sc_full.fit_transform(Xtrain.values.astype(np.float32))
X_te_full_s = sc_full.transform(Xtest.values.astype(np.float32))

for arr in [X_dl_tr, X_dl_va, X_dl_te, X_full_s, X_te_full_s]:
    np.nan_to_num(arr, copy=False)

X_tr_t = torch.from_numpy(X_dl_tr)
y_tr_t = torch.from_numpy(y_dl_tr).unsqueeze(1)
X_va_t = torch.from_numpy(X_dl_va)
y_va_t = torch.from_numpy(y_dl_va).unsqueeze(1)
X_te_t = torch.from_numpy(X_dl_te)
X_full_t = torch.from_numpy(X_full_s)
y_full_t = torch.from_numpy(ytrain.values.astype(np.float32)).unsqueeze(1)
X_te_full_t = torch.from_numpy(X_te_full_s)

device = torch.device(
    "mps" if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available() else "cpu"
)
INPUT_DIM = X_dl_tr.shape[1]
print(f"Device: {device},  Input dim: {INPUT_DIM}")
print(f"Train: {X_dl_tr.shape}, Val: {X_dl_va.shape}, Test: {X_dl_te.shape}\n")


# ═══════════════════════════════════════════════════════════════
#  2. АРХИТЕКТУРЫ  (24 шт.)
# ═══════════════════════════════════════════════════════════════

# ── 2.1  MLP ─────────────────────────────────────────────────

class MLP(nn.Module):
    def __init__(self, d_in, hidden_dims, dropout=0.3):
        super().__init__()
        layers = []
        prev = d_in
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                       nn.SiLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)


# ── 2.2  ResMLP ──────────────────────────────────────────────

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim),
        )
        self.act = nn.SiLU()
    def forward(self, x):
        return self.act(x + self.block(x))

class ResMLP(nn.Module):
    def __init__(self, d_in, hidden=256, n_blocks=3, dropout=0.3):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU())
        self.blocks = nn.Sequential(*[ResBlock(hidden, dropout) for _ in range(n_blocks)])
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        return self.head(self.blocks(self.proj(x)))


# ── 2.3  SNN ─────────────────────────────────────────────────

class SNN(nn.Module):
    def __init__(self, d_in, hidden_dims=[256, 128], alpha_dropout=0.1):
        super().__init__()
        layers = []
        prev = d_in
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.SELU(), nn.AlphaDropout(alpha_dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='linear')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    def forward(self, x):
        return self.net(x)


# ── 2.4  GatedMLP ───────────────────────────────────────────

class GatedBlock(nn.Module):
    def __init__(self, d_in, d_out, dropout):
        super().__init__()
        self.linear = nn.Linear(d_in, d_out * 2)
        self.bn = nn.BatchNorm1d(d_out)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        h = self.linear(x)
        h1, h2 = h.chunk(2, dim=-1)
        return self.drop(self.bn(h1 * torch.sigmoid(h2)))

class GatedMLP(nn.Module):
    def __init__(self, d_in, hidden=256, n_blocks=3, dropout=0.3):
        super().__init__()
        blocks = [GatedBlock(d_in, hidden, dropout)]
        for _ in range(n_blocks - 1):
            blocks.append(GatedBlock(hidden, hidden, dropout))
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        return self.head(self.blocks(x))


# ── 2.5  GANDALF ─────────────────────────────────────────────

class GANDALF(nn.Module):
    def __init__(self, d_in, hidden=256, n_blocks=3, dropout=0.3):
        super().__init__()
        self.gate_fc = nn.Linear(d_in, d_in)
        layers = [nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout)]
        for _ in range(n_blocks - 1):
            layers += [nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout)]
        self.dnn = nn.Sequential(*layers)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        gate = torch.sigmoid(self.gate_fc(x))
        return self.head(self.dnn(x * gate))


# ── 2.6  DAE-MLP ────────────────────────────────────────────

class DAEMLP(nn.Module):
    def __init__(self, d_in, hidden=256, latent=64, noise=0.3, dropout=0.3):
        super().__init__()
        self.noise = noise
        self.encoder = nn.Sequential(
            nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, latent), nn.BatchNorm1d(latent), nn.SiLU(),
        )
        self.decoder = nn.Sequential(nn.Linear(latent, hidden), nn.SiLU(), nn.Linear(hidden, d_in))
        self.reg_head = nn.Sequential(nn.Dropout(dropout), nn.Linear(latent, 32), nn.SiLU(), nn.Linear(32, 1))
        self.aux_loss = torch.tensor(0.0)
    def forward(self, x):
        x_in = x * (torch.rand_like(x) > self.noise).float() if self.training and self.noise > 0 else x
        z = self.encoder(x_in)
        if self.training:
            self.aux_loss = F.mse_loss(self.decoder(z), x)
        return self.reg_head(z)


# ── 2.7  CNN1D ───────────────────────────────────────────────

class CNN1D(nn.Module):
    def __init__(self, d_in, channels=[64, 128, 64], ks=5, dropout=0.3):
        super().__init__()
        layers = []
        in_ch = 1
        for ch in channels:
            layers += [nn.Conv1d(in_ch, ch, ks, padding=ks // 2),
                       nn.BatchNorm1d(ch), nn.SiLU(), nn.Dropout(dropout)]
            in_ch = ch
        self.conv = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(nn.Linear(channels[-1], 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.pool(self.conv(x))
        return self.head(x.squeeze(-1))


# ── 2.8  BiGRU ───────────────────────────────────────────────

class BiGRU(nn.Module):
    def __init__(self, d_in, hidden=64, n_layers=2, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(input_size=1, hidden_size=hidden, num_layers=n_layers,
                          batch_first=True, bidirectional=True,
                          dropout=dropout if n_layers > 1 else 0)
        self.head = nn.Sequential(nn.Linear(hidden * 2, 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        _, h = self.gru(x.unsqueeze(-1))
        return self.head(torch.cat([h[-2], h[-1]], dim=1))


# ── 2.9  FT-Transformer ─────────────────────────────────────

class FTTransformer(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, d_in + 1, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
              dim_feedforward=d_model * 4, dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        B = x.shape[0]
        tok = self.feat_emb(x.unsqueeze(-1))
        tok = torch.cat([self.cls.expand(B, -1, -1), tok], dim=1) + self.pos
        return self.head(self.transformer(tok)[:, 0])


# ── 2.10  TabTransformer ────────────────────────────────────

class TabTransformer(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, mlp_hidden=128, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.col_emb = nn.Parameter(torch.randn(1, d_in, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
              dim_feedforward=d_model * 4, dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.Linear(d_in * d_model + d_in, mlp_hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1))
    def forward(self, x):
        tok = self.feat_emb(x.unsqueeze(-1)) + self.col_emb
        out = self.transformer(tok)
        pooled = out.reshape(out.size(0), -1)
        return self.head(torch.cat([pooled, x], dim=1))


# ── 2.11  SAINT ──────────────────────────────────────────────

class SAINTBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.sa_norm = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ia_norm = nn.LayerNorm(d_model)
        self.inter_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model * 4),
                                 nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model * 4, d_model), nn.Dropout(dropout))
    def forward(self, x):
        h = self.sa_norm(x)
        h, _ = self.self_attn(h, h, h)
        x = x + h
        B, F_, D = x.shape
        if B > 1 and self.training:
            xt = x.permute(1, 0, 2)
            h2 = self.ia_norm(xt)
            h2, _ = self.inter_attn(h2, h2, h2)
            x = (xt + h2).permute(1, 0, 2)
        return x + self.ffn(x)

class SAINT(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, d_in + 1, d_model) * 0.02)
        self.blocks = nn.ModuleList([SAINTBlock(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        B = x.shape[0]
        tok = self.feat_emb(x.unsqueeze(-1))
        tok = torch.cat([self.cls.expand(B, -1, -1), tok], dim=1) + self.pos
        for blk in self.blocks:
            tok = blk(tok)
        return self.head(tok[:, 0])


# ── 2.12  AutoInt ────────────────────────────────────────────

class AutoIntLayer(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.W_res = nn.Linear(d_model, d_model, bias=False)
    def forward(self, x):
        h, _ = self.attn(x, x, x)
        return F.relu(self.norm(h + self.W_res(x)))

class AutoInt(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=3, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.layers = nn.ModuleList([AutoIntLayer(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.Linear(d_in * d_model, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 1))
    def forward(self, x):
        tok = self.feat_emb(x.unsqueeze(-1))
        for layer in self.layers:
            tok = layer(tok)
        return self.head(tok.reshape(tok.size(0), -1))


# ── 2.13  Trompt ─────────────────────────────────────────────

class TromptLayer(nn.Module):
    def __init__(self, d_in, d_model, n_heads, dropout):
        super().__init__()
        self.prompt = nn.Parameter(torch.randn(1, d_in, d_model) * 0.02)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_model * 2), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(d_model * 2, d_model))
        self.norm2 = nn.LayerNorm(d_model)
        self.gate = nn.Linear(d_model, 1)
    def forward(self, feat_emb, x_raw):
        B = feat_emb.shape[0]
        prompted = feat_emb + self.prompt.expand(B, -1, -1)
        h, _ = self.attn(prompted, prompted, prompted)
        h = self.norm1(prompted + h)
        h = self.norm2(h + self.ffn(h))
        weights = torch.softmax(self.gate(h).squeeze(-1), dim=1)
        return h, weights * x_raw

class Trompt(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=3, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.layers = nn.ModuleList([TromptLayer(d_in, d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.Linear(d_in, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, 1))
    def forward(self, x):
        tok = self.feat_emb(x.unsqueeze(-1))
        preds = []
        for layer in self.layers:
            tok, lp = layer(tok, x)
            preds.append(lp)
        return self.head(torch.stack(preds).mean(0))


# ── 2.14  ExcelFormer ───────────────────────────────────────

class ExcelFormer(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.importance = nn.Parameter(torch.zeros(d_in))
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, d_in + 1, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
              dim_feedforward=d_model * 4, dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        B = x.shape[0]
        mask = torch.sigmoid(self.importance)
        x_masked = x * mask
        tok = self.feat_emb(x_masked.unsqueeze(-1))
        tok = torch.cat([self.cls.expand(B, -1, -1), tok], dim=1) + self.pos
        return self.head(self.transformer(tok)[:, 0])


# ── 2.15  ARM-Net ────────────────────────────────────────────

class ARMNet(nn.Module):
    def __init__(self, d_in, n_exp=64, hidden=128, order=2, dropout=0.2):
        super().__init__()
        self.order = order
        self.exp_layers = nn.ModuleList([nn.Linear(d_in, n_exp) for _ in range(order)])
        self.gate = nn.Sequential(nn.Linear(d_in, n_exp * order), nn.Sigmoid())
        self.head = nn.Sequential(
            nn.BatchNorm1d(n_exp * order), nn.Linear(n_exp * order, hidden),
            nn.SiLU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, x):
        interactions = []
        for i, layer in enumerate(self.exp_layers):
            h = layer(x)
            if i > 0:
                h = h * interactions[-1]
            interactions.append(F.softplus(h))
        cat = torch.cat(interactions, dim=1)
        return self.head(cat * self.gate(x))


# ── 2.16  NODE ───────────────────────────────────────────────

class NODE(nn.Module):
    def __init__(self, d_in, n_trees=32, depth=4, dropout=0.0):
        super().__init__()
        self.n_trees = n_trees
        self.depth = depth
        n_leaves = 2 ** depth
        self.feat_weights = nn.Parameter(torch.randn(n_trees, depth, d_in) * 0.01)
        self.thresholds = nn.Parameter(torch.zeros(n_trees, depth))
        self.leaf_values = nn.Parameter(torch.randn(n_trees, n_leaves) * 0.01)
        self.drop = nn.Dropout(dropout)
        patterns = torch.zeros(n_leaves, depth)
        for i in range(n_leaves):
            for d in range(depth):
                patterns[i, d] = (i >> (depth - 1 - d)) & 1
        self.register_buffer("patterns", patterns)
    def forward(self, x):
        choices = torch.sigmoid(torch.einsum('bj,tdj->btd', x, self.feat_weights) - self.thresholds)
        c = choices.unsqueeze(2)
        p = self.patterns.unsqueeze(0).unsqueeze(0)
        leaf_probs = (c * p + (1 - c) * (1 - p)).prod(dim=-1)
        tree_out = (leaf_probs * self.leaf_values.unsqueeze(0)).sum(dim=-1)
        return self.drop(tree_out).mean(dim=-1, keepdim=True)


# ── 2.17  GRANDE ─────────────────────────────────────────────

class GRANDE(nn.Module):
    def __init__(self, d_in, n_trees=32, depth=4, dropout=0.0):
        super().__init__()
        self.n_trees = n_trees
        self.depth = depth
        n_leaves = 2 ** depth
        self.feat_logits = nn.Parameter(torch.randn(n_trees, depth, d_in) * 0.1)
        self.thresholds = nn.Parameter(torch.zeros(n_trees, depth))
        self.leaf_values = nn.Parameter(torch.randn(n_trees, n_leaves) * 0.01)
        self.tree_weights = nn.Parameter(torch.ones(n_trees) / n_trees)
        self.drop = nn.Dropout(dropout)
        patterns = torch.zeros(n_leaves, depth)
        for i in range(n_leaves):
            for d in range(depth):
                patterns[i, d] = (i >> (depth - 1 - d)) & 1
        self.register_buffer("patterns", patterns)
    def forward(self, x):
        feat_sel = F.softmax(self.feat_logits, dim=-1)
        proj = torch.einsum('bj,tdj->btd', x, feat_sel)
        choices = torch.sigmoid(10.0 * (proj - self.thresholds))
        c = choices.unsqueeze(2)
        p = self.patterns.unsqueeze(0).unsqueeze(0)
        leaf_probs = (c * p + (1 - c) * (1 - p)).prod(dim=-1)
        tree_out = (leaf_probs * self.leaf_values.unsqueeze(0)).sum(dim=-1)
        w = F.softmax(self.tree_weights, dim=0)
        return self.drop((tree_out * w).sum(dim=-1, keepdim=True))


# ── 2.18  Net-DNF ────────────────────────────────────────────

class NetDNF(nn.Module):
    def __init__(self, d_in, n_formulas=64, n_conjuncts=6, dropout=0.2):
        super().__init__()
        self.feat_sel = nn.Parameter(torch.randn(n_formulas, n_conjuncts, d_in) * 0.1)
        self.thresholds = nn.Parameter(torch.zeros(n_formulas, n_conjuncts))
        self.formula_w = nn.Linear(n_formulas, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        sel = torch.sigmoid(self.feat_sel)
        proj = torch.einsum('bj,fcj->bfc', x, sel)
        conjuncts = torch.sigmoid(proj - self.thresholds)
        formulas = conjuncts.prod(dim=-1)
        return self.formula_w(self.drop(formulas))


# ── 2.19  TabNet (simplified) ───────────────────────────────

class TabNet(nn.Module):
    def __init__(self, d_in, n_steps=3, n_d=64, n_a=64, relax=1.5, dropout=0.1):
        super().__init__()
        self.n_steps = n_steps
        self.relax = relax
        self.bn_init = nn.BatchNorm1d(d_in)
        self.shared_fc = nn.Linear(d_in, n_d + n_a)
        self.step_fcs = nn.ModuleList([nn.Linear(d_in, n_d + n_a) for _ in range(n_steps)])
        self.attn_fcs = nn.ModuleList([nn.Sequential(nn.Linear(n_a, d_in), nn.BatchNorm1d(d_in)) for _ in range(n_steps)])
        self.head = nn.Linear(n_d, 1)
        self.n_d = n_d
        self.drop = nn.Dropout(dropout)
        self.aux_loss = torch.tensor(0.0)
    def forward(self, x):
        x = self.bn_init(x)
        prior_scales = torch.ones(x.shape[0], x.shape[1], device=x.device)
        agg = torch.zeros(x.shape[0], self.n_d, device=x.device)
        entropy_loss = 0.0
        for i in range(self.n_steps):
            h = self.shared_fc(x * prior_scales) + self.step_fcs[i](x * prior_scales)
            h_d, h_a = h[:, :self.n_d], h[:, self.n_d:]
            h_d = F.silu(h_d)
            agg = agg + self.drop(h_d)
            attn = self.attn_fcs[i](h_a)
            attn = F.softmax(attn * prior_scales, dim=-1)
            prior_scales = prior_scales * (self.relax - attn)
            entropy_loss += (-attn * torch.log(attn + 1e-15)).mean()
        if self.training:
            self.aux_loss = entropy_loss / self.n_steps
        return self.head(agg)


# ── 2.20  TabR ───────────────────────────────────────────────

class TabR(nn.Module):
    def __init__(self, d_in, hidden=256, n_layers=3, k=32, dropout=0.3):
        super().__init__()
        self.k = k
        layers = []
        prev = d_in
        for i in range(n_layers):
            h = hidden if i == 0 else hidden // 2
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.SiLU(), nn.Dropout(dropout)]
            prev = h
        self.encoder = nn.Sequential(*layers)
        self.latent_dim = prev
        self.retrieval_head = nn.Sequential(nn.Linear(prev * 2, 128), nn.SiLU(), nn.Dropout(dropout), nn.Linear(128, 1))
        self.direct_head = nn.Linear(prev, 1)
        self._store_z = None
        self._store_y = None
    def build_store(self, x_train, y_train):
        self.eval()
        with torch.no_grad():
            self._store_z = self.encoder(x_train)
            self._store_y = y_train
    def forward(self, x):
        z = self.encoder(x)
        if self._store_z is not None and not self.training:
            dists = torch.cdist(z, self._store_z)
            _, idx = dists.topk(self.k, largest=False)
            nbr_z = self._store_z[idx]
            sim = -dists.gather(1, idx)
            attn = torch.softmax(sim, dim=1).unsqueeze(-1)
            context = (attn * nbr_z).sum(dim=1)
            return self.retrieval_head(torch.cat([z, context], dim=1))
        return self.direct_head(z)


# ── 2.21  GrowNet stages ────────────────────────────────────

class GrowNetStage(nn.Module):
    def __init__(self, d_in, hidden=32, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in + 1, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, 1))
    def forward(self, x, prev_pred):
        return self.net(torch.cat([x, prev_pred], dim=1))


# ── 2.22  SwitchTab ──────────────────────────────────────────

class SwitchTab(nn.Module):
    def __init__(self, d_in, d_enc=128, d_mutual=64, d_salient=64, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(d_in, d_enc), nn.BatchNorm1d(d_enc), nn.SiLU(), nn.Dropout(dropout))
        self.mutual_proj = nn.Linear(d_enc, d_mutual)
        self.salient_proj = nn.Linear(d_enc, d_salient)
        self.decoder = nn.Sequential(nn.Linear(d_mutual + d_salient, d_in))
        self.head = nn.Sequential(nn.Linear(d_mutual + d_salient, 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
        self.aux_loss = torch.tensor(0.0)
    def forward(self, x):
        h = self.encoder(x)
        mutual = self.mutual_proj(h)
        salient = self.salient_proj(h)
        rep = torch.cat([mutual, salient], dim=1)
        if self.training:
            self.aux_loss = F.mse_loss(self.decoder(rep), x)
        return self.head(rep)


# ── 2.23  PTaRL ──────────────────────────────────────────────

class PTaRL(nn.Module):
    def __init__(self, d_in, n_prototypes=16, d_latent=128, hidden=256, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, d_latent), nn.BatchNorm1d(d_latent), nn.SiLU())
        self.prototypes = nn.Parameter(torch.randn(n_prototypes, d_latent) * 0.1)
        self.head = nn.Sequential(nn.Linear(d_latent * 2, 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        z = self.encoder(x)
        sim = F.cosine_similarity(z.unsqueeze(1), self.prototypes.unsqueeze(0), dim=-1)
        attn = F.softmax(sim * 10, dim=1)
        proto_rep = (attn.unsqueeze(-1) * self.prototypes.unsqueeze(0)).sum(dim=1)
        return self.head(torch.cat([z, proto_rep], dim=1))


# ── 2.24  DCN v2 ────────────────────────────────────────────

class CrossLayer(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.W = nn.Linear(d, d, bias=False)
        self.b = nn.Parameter(torch.zeros(d))
    def forward(self, x0, x):
        return x0 * self.W(x) + self.b + x

class DCNv2(nn.Module):
    def __init__(self, d_in, n_cross=3, hidden=256, dropout=0.3):
        super().__init__()
        self.cross_layers = nn.ModuleList([CrossLayer(d_in) for _ in range(n_cross)])
        self.deep = nn.Sequential(
            nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.BatchNorm1d(hidden // 2), nn.SiLU(), nn.Dropout(dropout))
        self.head = nn.Linear(d_in + hidden // 2, 1)
    def forward(self, x):
        x_cross = x
        for cl in self.cross_layers:
            x_cross = cl(x, x_cross)
        return self.head(torch.cat([x_cross, self.deep(x)], dim=1))


# ═══════════════════════════════════════════════════════════════
#  3. ФУНКЦИИ ОБУЧЕНИЯ
# ═══════════════════════════════════════════════════════════════

def train_model(model, epochs=300, lr=1e-3, wd=1e-4, bs=256,
                patience=30, aux_w=0.0, trial=None):
    """Обучение с early stopping. Возвращает (model, val_mae, epochs_used)."""
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", patience=10, factor=0.5, min_lr=1e-6)
    loss_fn = nn.L1Loss()
    ds = TensorDataset(X_tr_t, y_tr_t)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    X_v, y_v = X_va_t.to(device), y_va_t.to(device)

    best_val, best_state, wait = float("inf"), None, 0
    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            if aux_w > 0 and hasattr(model, "aux_loss"):
                loss = loss + aux_w * model.aux_loss
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            v_mae = loss_fn(model(X_v), y_v).item()
        sched.step(v_mae)
        if v_mae < best_val:
            best_val = v_mae
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
        if trial is not None:
            trial.report(v_mae, ep)
            if trial.should_prune():
                raise optuna.TrialPruned()
    model.load_state_dict(best_state)
    return model, best_val, ep


def train_grownet(n_stages=5, hidden=32, lr=0.01, wd=1e-4,
                  step_size=0.1, bs=256, patience=30, dropout=0.1, trial=None):
    """Gradient boosting с NN base learners. Возвращает (stages, val_mae, total_epochs)."""
    stages = []
    X_v, y_v = X_va_t.to(device), y_va_t.to(device)
    ds = TensorDataset(X_tr_t, y_tr_t)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    loss_fn = nn.L1Loss()

    # Предсказание ансамбля на train/val
    def ensemble_pred(x, stgs):
        p = torch.zeros(x.size(0), 1, device=x.device)
        for s in stgs:
            s.eval()
            p = p + step_size * s(x, p.detach())
        return p

    best_val, best_n = float("inf"), 0
    total_ep = 0
    for stage_idx in range(n_stages):
        model = GrowNetStage(INPUT_DIM, hidden, dropout).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", patience=10, factor=0.5, min_lr=1e-6)
        best_stage_val, best_stage_state, wait = float("inf"), None, 0
        for ep in range(1, 201):
            model.train()
            for xb, yb in dl:
                xb, yb = xb.to(device), yb.to(device)
                with torch.no_grad():
                    prev = ensemble_pred(xb, stages)
                correction = model(xb, prev)
                loss = loss_fn(prev + step_size * correction, yb)
                opt.zero_grad(); loss.backward(); opt.step()
            model.eval()
            with torch.no_grad():
                prev_v = ensemble_pred(X_v, stages)
                v_pred = prev_v + step_size * model(X_v, prev_v)
                v_mae = loss_fn(v_pred, y_v).item()
            sched.step(v_mae)
            if v_mae < best_stage_val:
                best_stage_val = v_mae
                best_stage_state = {k: v.clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    break
            total_ep += 1
        model.load_state_dict(best_stage_state)
        stages.append(model)
        # Проверяем ансамбль
        with torch.no_grad():
            ens_val = loss_fn(ensemble_pred(X_v, stages), y_v).item()
        if ens_val < best_val:
            best_val = ens_val
            best_n = len(stages)
        if trial is not None:
            trial.report(ens_val, stage_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()
    stages = stages[:best_n]
    return stages, best_val, total_ep


def eval_on_test(model, name=""):
    """Evaluate model on holdout test."""
    model.eval()
    with torch.no_grad():
        pred = model(X_te_t.to(device)).cpu().numpy().flatten()
    mae = mean_absolute_error(ytest, pred)
    rmse = np.sqrt(mean_squared_error(ytest, pred))
    mdae = median_absolute_error(ytest, pred)
    return mae, rmse, mdae


def eval_grownet_on_test(stages, step_size=0.1):
    """Evaluate GrowNet ensemble on holdout test."""
    X_t = X_te_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    mae = mean_absolute_error(ytest, pred)
    rmse = np.sqrt(mean_squared_error(ytest, pred))
    mdae = median_absolute_error(ytest, pred)
    return mae, rmse, mdae


print("24 архитектуры + train_model + train_grownet определены.")


In [ ]:
# Дополнительные тензоры/функции для полного holdout (stress-test),
# чтобы не ломать исходные train/eval-хелперы из исходного ноутбука.

X_dl_te_full = sc.transform(Xtest_full.values.astype(np.float32))
np.nan_to_num(X_dl_te_full, copy=False)
X_te_stress_t = torch.from_numpy(X_dl_te_full)

X_te_fullscale_stress = sc_full.transform(Xtest_full.values.astype(np.float32))
np.nan_to_num(X_te_fullscale_stress, copy=False)
X_te_fullscale_stress_t = torch.from_numpy(X_te_fullscale_stress)

def clear_device_cache():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        torch.mps.empty_cache()

def calc_reg_metrics(y_true, pred):
    return {
        "MAE": float(mean_absolute_error(y_true, pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, pred))),
        "MdAE": float(median_absolute_error(y_true, pred)),
    }

def eval_on_typical_holdout(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te_t.to(device)).cpu().numpy().flatten()
    return calc_reg_metrics(ytest_typical, pred)

def eval_on_full_holdout(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te_stress_t.to(device)).cpu().numpy().flatten()
    return calc_reg_metrics(ytest_full, pred)

def eval_grownet_on_typical_holdout(stages, step_size=0.1):
    X_t = X_te_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    return calc_reg_metrics(ytest_typical, pred)

def eval_grownet_on_full_holdout(stages, step_size=0.1):
    X_t = X_te_stress_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    return calc_reg_metrics(ytest_full, pred)

def eval_tabpfn_holdouts(tabpfn_model, use_full_train_scaler=False):
    if use_full_train_scaler:
        X_typ = X_te_full_t.numpy()
        X_full = X_te_fullscale_stress_t.numpy()
    else:
        X_typ = X_te_t.numpy()
        X_full = X_te_stress_t.numpy()
    pred_typ = tabpfn_model.predict(X_typ)
    pred_full = tabpfn_model.predict(X_full)
    return calc_reg_metrics(ytest_typical, pred_typ), calc_reg_metrics(ytest_full, pred_full)

def train_model_fulltrain(model, epochs=100, lr=1e-3, wd=1e-4, bs=256, aux_w=0.0):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1), eta_min=1e-6)
    loss_fn = nn.L1Loss()
    ds = TensorDataset(X_full_t, y_full_t)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    for ep in range(epochs):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            if aux_w > 0 and hasattr(model, "aux_loss"):
                loss = loss + aux_w * model.aux_loss
            opt.zero_grad()
            loss.backward()
            opt.step()
        sched.step()
    return model

def eval_fulltrain_model_typical(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te_full_t.to(device)).cpu().numpy().flatten()
    return calc_reg_metrics(ytest_typical, pred)

def eval_fulltrain_model_full(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te_fullscale_stress_t.to(device)).cpu().numpy().flatten()
    return calc_reg_metrics(ytest_full, pred)

def train_grownet_fulltrain(n_stages=5, hidden=32, lr=0.01, wd=1e-4,
                            step_size=0.1, bs=256, epochs_per_stage=50, dropout=0.1):
    stages = []
    ds = TensorDataset(X_full_t, y_full_t)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    loss_fn = nn.L1Loss()

    def ensemble_pred(x, stgs):
        p = torch.zeros(x.size(0), 1, device=x.device)
        for s in stgs:
            s.eval()
            p = p + step_size * s(x, p.detach())
        return p

    for stage_idx in range(n_stages):
        model = GrowNetStage(INPUT_DIM, hidden, dropout).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs_per_stage, 1), eta_min=1e-6)

        for ep in range(epochs_per_stage):
            model.train()
            for xb, yb in dl:
                xb, yb = xb.to(device), yb.to(device)
                with torch.no_grad():
                    prev = ensemble_pred(xb, stages)
                correction = model(xb, prev)
                loss = loss_fn(prev + step_size * correction, yb)
                opt.zero_grad()
                loss.backward()
                opt.step()
            sched.step()

        stages.append(model)
    return stages

def eval_fulltrain_grownet_typical(stages, step_size=0.1):
    X_t = X_te_full_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    return calc_reg_metrics(ytest_typical, pred)

def eval_fulltrain_grownet_full(stages, step_size=0.1):
    X_t = X_te_fullscale_stress_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    return calc_reg_metrics(ytest_full, pred)

print("Full-holdout helpers defined.")

In [ ]:
def build_arch_from_params(arch_name: str, bp: dict):
    if arch_name == "MLP":
        dims = [max(int(bp["first_hidden"]) // (2 ** i), 32) for i in range(int(bp["n_layers"]))]
        return MLP(INPUT_DIM, dims, bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "ResMLP":
        return ResMLP(INPUT_DIM, bp["hidden"], bp["n_blocks"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "SNN":
        dims = [max(int(bp["first_hidden"]) // (2 ** i), 32) for i in range(int(bp["n_layers"]))]
        return SNN(INPUT_DIM, dims, bp["alpha_dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "GatedMLP":
        return GatedMLP(INPUT_DIM, bp["hidden"], bp["n_blocks"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "GANDALF":
        return GANDALF(INPUT_DIM, bp["hidden"], bp["n_blocks"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "DAE-MLP":
        return DAEMLP(INPUT_DIM, bp["hidden"], bp["latent"], bp["noise"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "CNN1D":
        nc = int(bp["n_conv"]); bc = int(bp["base_ch"])
        chs = [bc * (2 ** i) for i in range(nc // 2 + 1)]
        chs += [bc * (2 ** i) for i in range(nc // 2 - 1, -1, -1)]
        chs = chs[:nc]
        return CNN1D(INPUT_DIM, chs, bp["ks"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "BiGRU":
        return BiGRU(INPUT_DIM, bp["hidden"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "FT-Transformer":
        return FTTransformer(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "TabTransformer":
        return TabTransformer(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["mlp_hidden"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "SAINT":
        return SAINT(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "AutoInt":
        return AutoInt(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "Trompt":
        return Trompt(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "ExcelFormer":
        return ExcelFormer(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "ARM-Net":
        return ARMNet(INPUT_DIM, bp["n_exp"], bp["hidden"], bp["order"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "NODE":
        return NODE(INPUT_DIM, bp["n_trees"], bp["depth"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "GRANDE":
        return GRANDE(INPUT_DIM, bp["n_trees"], bp["depth"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "Net-DNF":
        return NetDNF(INPUT_DIM, bp["n_formulas"], bp["n_conjuncts"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "TabNet":
        return TabNet(INPUT_DIM, bp["n_steps"], bp["n_d"], bp["n_a"], bp["relax"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "TabR":
        return TabR(INPUT_DIM, bp["hidden"], bp["n_layers"], bp["k"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "SwitchTab":
        return SwitchTab(INPUT_DIM, bp["d_enc"], bp["d_mutual"], bp["d_salient"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "PTaRL":
        return PTaRL(INPUT_DIM, bp["n_prototypes"], bp["d_latent"], bp["hidden"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "DCNv2":
        return DCNv2(INPUT_DIM, bp["n_cross"], bp["hidden"], bp["dropout"]), bp.get("aux_w", 0.0)
    else:
        raise KeyError(f"Unknown architecture: {arch_name}")

def serialize_params(params: dict):
    clean = {}
    for k, v in params.items():
        if isinstance(v, (np.integer,)):
            clean[k] = int(v)
        elif isinstance(v, (np.floating,)):
            clean[k] = float(v)
        else:
            clean[k] = v
    return clean

In [ ]:

import os
import re
import ast
import json
import math
import time
import gc
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import nnls, minimize
from sklearn.base import clone
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV, BayesianRidge, HuberRegressor, QuantileRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error

try:
    from xgboost import XGBRegressor
    HAS_XGB_FOR_STACKING = True
except Exception:
    HAS_XGB_FOR_STACKING = False

def sanitize_name(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", name)

def calc_reg_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(float)
    y_pred = np.asarray(y_pred).astype(float)
    y_pred = np.clip(y_pred, 0, None)
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MdAE": float(median_absolute_error(y_true, y_pred)),
    }

def cleanup_memory():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

def _normalize_param_string(raw: str) -> str:
    s = str(raw).strip()
    # common CSV / repr artefacts from pandas
    s = re.sub(r'\bNaN\b', 'None', s)
    s = re.sub(r'\bnan\b', 'None', s)
    s = re.sub(r'\bnull\b', 'None', s, flags=re.IGNORECASE)
    s = re.sub(r'\btrue\b', 'True', s, flags=re.IGNORECASE)
    s = re.sub(r'\bfalse\b', 'False', s, flags=re.IGNORECASE)
    return s

def parse_best_params(raw):
    if isinstance(raw, dict):
        bp = raw
    elif raw is None or (isinstance(raw, float) and pd.isna(raw)):
        bp = {}
    else:
        s = _normalize_param_string(raw)
        try:
            # works when params are valid JSON
            bp = json.loads(s)
        except Exception:
            try:
                # works when params are python-dict repr strings
                bp = ast.literal_eval(s)
            except Exception as e:
                raise ValueError(f"Could not parse params: {raw}") from e
    return serialize_params(bp)

def pick_model_pool(tuned_frame: pd.DataFrame):
    tmp = tuned_frame.copy()
    tmp["best_params_parsed"] = tmp["best_params"].apply(parse_best_params)
    tmp = tmp.sort_values("selected_val_MAE", kind="stable").reset_index(drop=True)
    if USE_ALL_TUNED_MODELS:
        return tmp
    if MANUAL_MODEL_POOL:
        return tmp[tmp["model"].isin(MANUAL_MODEL_POOL)].reset_index(drop=True)
    return tmp

POOL_DF = pick_model_pool(tuned_df)
SELECTED_MODELS = POOL_DF["model"].tolist()
print(f"Models selected for ensemble rebuild ({len(SELECTED_MODELS)}):")
print(SELECTED_MODELS)

def _pred_path(kind: str, model_name: str) -> Path:
    return PRED_CACHE_DIR / kind / f"{sanitize_name(model_name)}.npy"

def _save_pred(kind: str, model_name: str, arr):
    np.save(_pred_path(kind, model_name), np.asarray(arr, dtype=np.float32))

def _load_pred(kind: str, model_name: str):
    path = _pred_path(kind, model_name)
    if not path.exists():
        return None
    return np.load(path)

def _have_all_cached(model_name: str) -> bool:
    needed = [
        _pred_path("split_val", model_name),
        _pred_path("split_test_typical", model_name),
        _pred_path("split_test_full", model_name),
        _pred_path("fullfit_test_typical", model_name),
        _pred_path("fullfit_test_full", model_name),
    ]
    return all(p.exists() for p in needed)

def _torch_pred(model, X_tensor):
    model.eval()
    with torch.no_grad():
        pred = model(X_tensor.to(device)).cpu().numpy().reshape(-1)
    return np.clip(pred, 0, None)

def _grownet_pred(stages, X_tensor, step_size=0.1):
    X_t = X_tensor.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().reshape(-1)
    return np.clip(pred, 0, None)

def fit_and_predict_tuned_model(row: pd.Series):
    arch = row["model"]
    best_params = parse_best_params(row["best_params"])
    epochs_hint = max(1, int(row.get("epochs", 100)))

    print(f"\n=== Rebuilding {arch} ===")
    print(best_params)

    seed_everything(SEED)
    t0 = time.time()

    if arch == "TabPFN":
        # Важно: здесь используется TabPFN-2.5 через Hugging Face auth, а не дефолтный browser-login flow TabPFN-2.6.
        if not globals().get("HAS_TABPFN", False):
            raise RuntimeError("TabPFN выбран, но пакет tabpfn недоступен.")
        pfn_device = "cuda" if torch.cuda.is_available() else "cpu"
        max_rows = int(best_params.get("max_rows", 3000))
        n_estimators = int(best_params.get("n_estimators", 8))

        # split-fit
        tabpfn = make_tabpfn_regressor(device=pfn_device, n_estimators=n_estimators)
        X_pfn_tr = X_dl_tr.copy()
        y_pfn_tr = y_dl_tr.copy()
        if len(X_pfn_tr) > max_rows:
            idx = np.random.RandomState(SEED).choice(len(X_pfn_tr), max_rows, replace=False)
            X_pfn_tr = X_pfn_tr[idx]
            y_pfn_tr = y_pfn_tr[idx]
        tabpfn.fit(X_pfn_tr, y_pfn_tr)
        pred_val = np.clip(tabpfn.predict(X_dl_va), 0, None)
        pred_test_typ = np.clip(tabpfn.predict(X_dl_te), 0, None)
        pred_test_full = np.clip(tabpfn.predict(X_te_stress_t.numpy()), 0, None)

        # full-fit
        tabpfn_full = make_tabpfn_regressor(device=pfn_device, n_estimators=n_estimators)
        X_pfn_full = X_full_s.copy()
        y_pfn_full = ytrain.values.astype(np.float32)
        if len(X_pfn_full) > max_rows:
            idx = np.random.RandomState(SEED).choice(len(X_pfn_full), max_rows, replace=False)
            X_pfn_full = X_pfn_full[idx]
            y_pfn_full = y_pfn_full[idx]
        tabpfn_full.fit(X_pfn_full, y_pfn_full)
        pred_fullfit_typ = np.clip(tabpfn_full.predict(X_te_full_t.numpy()), 0, None)
        pred_fullfit_full = np.clip(tabpfn_full.predict(X_te_fullscale_stress_t.numpy()), 0, None)

    elif arch == "GrowNet":
        # split-fit
        stages, _, _ = train_grownet(
            n_stages=int(best_params["n_stages"]),
            hidden=int(best_params["hidden"]),
            lr=float(best_params["lr"]),
            wd=float(best_params["wd"]),
            step_size=float(best_params["step_size"]),
            bs=int(best_params["bs"]),
            patience=30,
            dropout=float(best_params["dropout"]),
        )
        pred_val = _grownet_pred(stages, X_va_t, step_size=float(best_params["step_size"]))
        pred_test_typ = _grownet_pred(stages, X_te_t, step_size=float(best_params["step_size"]))
        pred_test_full = _grownet_pred(stages, X_te_stress_t, step_size=float(best_params["step_size"]))

        # full-fit
        n_stages = int(best_params["n_stages"])
        epochs_per_stage = max(20, math.ceil(epochs_hint / max(n_stages, 1)))
        stages_full = train_grownet_fulltrain(
            n_stages=n_stages,
            hidden=int(best_params["hidden"]),
            lr=float(best_params["lr"]),
            wd=float(best_params["wd"]),
            step_size=float(best_params["step_size"]),
            bs=int(best_params["bs"]),
            epochs_per_stage=epochs_per_stage,
            dropout=float(best_params["dropout"]),
        )
        pred_fullfit_typ = _grownet_pred(stages_full, X_te_full_t, step_size=float(best_params["step_size"]))
        pred_fullfit_full = _grownet_pred(stages_full, X_te_fullscale_stress_t, step_size=float(best_params["step_size"]))

    else:
        # split-fit
        model, aux_w = build_arch_from_params(arch, best_params)
        model, _, _ = train_model(
            model,
            epochs=epochs_hint,
            lr=float(best_params["lr"]),
            wd=float(best_params["wd"]),
            bs=int(best_params["bs"]),
            patience=30,
            aux_w=float(aux_w),
        )
        if arch == "TabR":
            model.build_store(X_tr_t.to(device), y_tr_t.to(device))
        pred_val = _torch_pred(model, X_va_t)
        pred_test_typ = _torch_pred(model, X_te_t)
        pred_test_full = _torch_pred(model, X_te_stress_t)

        # full-fit
        full_model, aux_w_full = build_arch_from_params(arch, best_params)
        full_model = train_model_fulltrain(
            full_model,
            epochs=epochs_hint,
            lr=float(best_params["lr"]),
            wd=float(best_params["wd"]),
            bs=int(best_params["bs"]),
            aux_w=float(aux_w_full),
        )
        if arch == "TabR":
            full_model.build_store(X_full_t.to(device), y_full_t.to(device))
        pred_fullfit_typ = _torch_pred(full_model, X_te_full_t)
        pred_fullfit_full = _torch_pred(full_model, X_te_fullscale_stress_t)

    elapsed = time.time() - t0
    print(f"{arch} rebuilt in {elapsed:.1f}s")

    return {
        "split_val": pred_val,
        "split_test_typical": pred_test_typ,
        "split_test_full": pred_test_full,
        "fullfit_test_typical": pred_fullfit_typ,
        "fullfit_test_full": pred_fullfit_full,
        "time_s": elapsed,
    }

def build_prediction_matrices(model_pool_df: pd.DataFrame):
    rebuild_log = []
    for _, row in model_pool_df.iterrows():
        model_name = row["model"]
        use_cache = RESUME_IF_PREDICTIONS_EXIST and (not FORCE_REBUILD_BASE_PREDICTIONS) and _have_all_cached(model_name)
        if use_cache:
            print(f"Using cached predictions for {model_name}")
            rebuild_log.append({"model": model_name, "status": "cache"})
            continue
        try:
            preds = fit_and_predict_tuned_model(row)
            for kind, arr in preds.items():
                if kind == "time_s":
                    continue
                _save_pred(kind, model_name, arr)
            rebuild_log.append({"model": model_name, "status": "rebuilt", "time_s": preds["time_s"]})
        except Exception as e:
            rebuild_log.append({"model": model_name, "status": f"failed:{type(e).__name__}", "error": str(e)})
            print(f"FAILED for {model_name}: {type(e).__name__}: {e}")
        finally:
            cleanup_memory()
    rebuild_log_df = pd.DataFrame(rebuild_log)
    return rebuild_log_df

def load_prediction_frame(kind: str, model_names):
    cols = {}
    for m in model_names:
        arr = _load_pred(kind, m)
        if arr is not None:
            cols[m] = arr
    if not cols:
        return pd.DataFrame()
    return pd.DataFrame(cols)

def get_base_leaderboard_from_predictions(val_pred_df: pd.DataFrame):
    rows = []
    for m in val_pred_df.columns:
        rows.append({
            "model": m,
            "val_MAE_rebuilt": float(mean_absolute_error(y_dl_va, val_pred_df[m].values)),
            "val_RMSE_rebuilt": float(np.sqrt(mean_squared_error(y_dl_va, val_pred_df[m].values))),
        })
    return pd.DataFrame(rows).sort_values("val_MAE_rebuilt", kind="stable").reset_index(drop=True)


## Add tuned ML models to the same ensemble pool

Ниже добавляется восстановление tuned **ML** моделей из `final_summary.json` / `final_summary.csv`.
Итоговый ensemble-search работает уже по объединённому пулу **DL + ML** моделей.


In [ ]:

# =========================
# Load tuned ML summary + ML rebuild helpers
# =========================
ML_USE_ALL_TUNED_MODELS = True
MANUAL_ML_MODEL_POOL = []   # если нужно ограничить ML-пул вручную

ML_TUNED_SUMMARY_CANDIDATES = [
    Path("./final_summary.json"),
    Path("./final_summary.csv"),
    Path("/content/final_summary.json"),
    Path("/content/final_summary.csv"),
    Path("/mnt/data/final_summary.json"),
    Path("/mnt/data/final_summary.csv"),
]

if IN_COLAB and USE_GOOGLE_DRIVE:
    ML_TUNED_SUMMARY_CANDIDATES.extend([
        DRIVE_PROJECT_DIR / "final_summary.json",
        DRIVE_PROJECT_DIR / "data" / "final_summary.json",
        DRIVE_PROJECT_DIR / "artifacts_tuning_final" / "final_summary.json",
        DRIVE_PROJECT_DIR / "artifacts_tuning_final" / "final_summary.csv",
        DRIVE_PROJECT_DIR / "artifacts_tuning_pointfix" / "final_summary.csv",
    ])

ML_TUNED_SUMMARY_CANDIDATES = [p for p in ML_TUNED_SUMMARY_CANDIDATES if p is not None]
ML_TUNED_SUMMARY_PATH = next((p for p in ML_TUNED_SUMMARY_CANDIDATES if p.exists()), None)
if ML_TUNED_SUMMARY_PATH is None:
    raise FileNotFoundError(
        "Не найден final_summary.json/final_summary.csv с tuned ML-моделями."
    )

if ML_TUNED_SUMMARY_PATH.suffix.lower() == ".json":
    ml_tuned_df = pd.DataFrame(json.load(open(ML_TUNED_SUMMARY_PATH, "r", encoding="utf-8")))
else:
    ml_tuned_df = pd.read_csv(ML_TUNED_SUMMARY_PATH)

if "params" not in ml_tuned_df.columns:
    raise ValueError("В ML tuned summary нет колонки `params`.")

print(f"Loaded ML tuned summary from: {ML_TUNED_SUMMARY_PATH}")
display(ml_tuned_df[["model", "selected_from", "cv_MAE", "holdout_typical_MAE", "holdout_full_MAE"]])

# ML inner split must match DL split, so that validation predictions are aligned
X_blend_tr = Xtrain.iloc[:val_cut].copy()
y_blend_tr = ytrain.iloc[:val_cut].copy()
X_blend_val = Xtrain.iloc[val_cut:].copy()
y_blend_val = ytrain.iloc[val_cut:].copy()

def parse_best_params_ml(raw):
    # reuse generic parser from DL ensemble notebook
    return parse_best_params(raw)

def make_ml_estimator(model_name: str, params: dict):
    p = serialize_params(params)

    if model_name == "BayRidge":
        model = BayesianRidge(
            alpha_1=float(p.get("alpha_1", 1e-6)),
            alpha_2=float(p.get("alpha_2", 1e-6)),
            lambda_1=float(p.get("lambda_1", 1e-6)),
            lambda_2=float(p.get("lambda_2", 1e-6)),
            max_iter=int(p.get("max_iter", 300)),
            tol=float(p.get("tol", 1e-3)),
            fit_intercept=bool(p.get("fit_intercept", True)),
            compute_score=bool(p.get("compute_score", False)),
            verbose=bool(p.get("verbose", False)),
        )
        return Pipeline([("scaler", StandardScaler()), ("model", model)])

    if model_name == "Lasso":
        model = Lasso(
            alpha=float(p.get("alpha", 1.0)),
            fit_intercept=bool(p.get("fit_intercept", True)),
            max_iter=int(p.get("max_iter", 100000)),
            tol=float(p.get("tol", 1e-4)),
            random_state=p.get("random_state", SEED),
            selection=p.get("selection", "cyclic"),
            warm_start=bool(p.get("warm_start", False)),
            positive=bool(p.get("positive", False)),
            precompute=bool(p.get("precompute", False)),
        )
        return Pipeline([("scaler", StandardScaler()), ("model", model)])

    if model_name == "Elastic":
        model = ElasticNet(
            alpha=float(p.get("alpha", 1.0)),
            l1_ratio=float(p.get("l1_ratio", 0.5)),
            fit_intercept=bool(p.get("fit_intercept", True)),
            max_iter=int(p.get("max_iter", 100000)),
            tol=float(p.get("tol", 1e-4)),
            random_state=p.get("random_state", SEED),
            selection=p.get("selection", "cyclic"),
            warm_start=bool(p.get("warm_start", False)),
            positive=bool(p.get("positive", False)),
            precompute=bool(p.get("precompute", False)),
        )
        return Pipeline([("scaler", StandardScaler()), ("model", model)])

    if model_name == "Ridge":
        model = Ridge(
            alpha=float(p.get("alpha", 1.0)),
            fit_intercept=bool(p.get("fit_intercept", True)),
            copy_X=bool(p.get("copy_X", True)),
            max_iter=None if p.get("max_iter") is None else int(p.get("max_iter")),
            tol=float(p.get("tol", 1e-4)),
            solver=p.get("solver", "auto"),
            positive=bool(p.get("positive", False)),
            random_state=p.get("random_state", None),
        )
        return Pipeline([("scaler", StandardScaler()), ("model", model)])

    if model_name == "Hub-Reg":
        model = HuberRegressor(
            epsilon=float(p.get("epsilon", 1.35)),
            alpha=float(p.get("alpha", 1e-4)),
            fit_intercept=bool(p.get("fit_intercept", True)),
            max_iter=int(p.get("max_iter", 1000)),
            tol=float(p.get("tol", 1e-5)),
            warm_start=bool(p.get("warm_start", False)),
        )
        return Pipeline([("scaler", StandardScaler()), ("model", model)])

    if model_name == "Gboost-Reg":
        model = GradientBoostingRegressor(
            loss=p.get("loss", "absolute_error"),
            learning_rate=float(p.get("learning_rate", 0.1)),
            n_estimators=int(p.get("n_estimators", 100)),
            subsample=float(p.get("subsample", 1.0)),
            criterion=p.get("criterion", "friedman_mse"),
            min_samples_split=int(p.get("min_samples_split", 2)),
            min_samples_leaf=int(p.get("min_samples_leaf", 1)),
            min_weight_fraction_leaf=float(p.get("min_weight_fraction_leaf", 0.0)),
            max_depth=int(p.get("max_depth", 3)),
            min_impurity_decrease=float(p.get("min_impurity_decrease", 0.0)),
            init=p.get("init", None),
            random_state=p.get("random_state", SEED),
            max_features=p.get("max_features", None),
            alpha=float(p.get("alpha", 0.9)),
            verbose=int(p.get("verbose", 0)),
            max_leaf_nodes=p.get("max_leaf_nodes", None),
            warm_start=bool(p.get("warm_start", False)),
            validation_fraction=float(p.get("validation_fraction", 0.1)),
            n_iter_no_change=p.get("n_iter_no_change", None),
            tol=float(p.get("tol", 1e-4)),
            ccp_alpha=float(p.get("ccp_alpha", 0.0)),
        )
        return model

    if model_name == "XGB_reg":
        if not HAS_XGBOOST:
            raise RuntimeError("xgboost недоступен в окружении.")
        xgb_params = {
            "objective": p.get("objective", "reg:absoluteerror"),
            "eval_metric": p.get("eval_metric", "mae"),
            "n_estimators": int(p.get("n_estimators", 500)),
            "learning_rate": float(p.get("learning_rate", 0.05)),
            "max_depth": int(p.get("max_depth", 3)),
            "min_child_weight": float(p.get("min_child_weight", 1.0)),
            "subsample": float(p.get("subsample", 1.0)),
            "colsample_bytree": float(p.get("colsample_bytree", 1.0)),
            "gamma": float(p.get("gamma", 0.0)),
            "reg_alpha": float(p.get("reg_alpha", 0.0)),
            "reg_lambda": float(p.get("reg_lambda", 1.0)),
            "max_bin": int(p.get("max_bin", 256)),
            "tree_method": p.get("tree_method", "hist"),
            "n_jobs": int(p.get("n_jobs", -1)),
            "random_state": p.get("random_state", SEED),
        }
        return XGBRegressor(**xgb_params)

    raise KeyError(model_name)

ml_tuned_df["best_params_parsed"] = ml_tuned_df["params"].apply(parse_best_params_ml)
ml_tuned_df = ml_tuned_df.sort_values(["cv_MAE", "holdout_typical_MAE"], kind="stable").reset_index(drop=True)

if ML_USE_ALL_TUNED_MODELS:
    ML_POOL_DF = ml_tuned_df.copy().reset_index(drop=True)
elif MANUAL_ML_MODEL_POOL:
    ML_POOL_DF = ml_tuned_df[ml_tuned_df["model"].isin(MANUAL_ML_MODEL_POOL)].copy().reset_index(drop=True)
else:
    ML_POOL_DF = ml_tuned_df.copy().reset_index(drop=True)

ML_SELECTED_MODELS = ML_POOL_DF["model"].tolist()
print(f"Selected tuned ML models ({len(ML_SELECTED_MODELS)}):")
print(ML_SELECTED_MODELS)

def fit_and_predict_tuned_ml_model(row: pd.Series):
    model_name = row["model"]
    best_params = row["best_params_parsed"]

    print(f"\n=== Rebuilding ML {model_name} ===")
    print(best_params)
    t0 = time.time()

    if RESUME_IF_PREDICTIONS_EXIST and not FORCE_REBUILD_BASE_PREDICTIONS and _have_all_cached(model_name):
        print("Using cached predictions.")
        return {"model": model_name, "group": "ML", "status": "cached", "elapsed_sec": 0.0}

    seed_everything(SEED)

    # split-fit (for blend validation)
    model_split = make_ml_estimator(model_name, best_params)
    model_split.fit(X_blend_tr, y_blend_tr)
    pred_val = clip_pred(model_split.predict(X_blend_val))
    pred_test_typ = clip_pred(model_split.predict(Xtest_typical))
    pred_test_full = clip_pred(model_split.predict(Xtest_full))

    # full-fit (for final holdout refit)
    model_full = make_ml_estimator(model_name, best_params)
    model_full.fit(Xtrain, ytrain)
    pred_test_typ_fullfit = clip_pred(model_full.predict(Xtest_typical))
    pred_test_full_fullfit = clip_pred(model_full.predict(Xtest_full))

    _save_pred("split_val", model_name, pred_val)
    _save_pred("split_test_typical", model_name, pred_test_typ)
    _save_pred("split_test_full", model_name, pred_test_full)
    _save_pred("fullfit_test_typical", model_name, pred_test_typ_fullfit)
    _save_pred("fullfit_test_full", model_name, pred_test_full_fullfit)

    elapsed = time.time() - t0
    return {
        "model": model_name,
        "group": "ML",
        "status": "rebuilt",
        "elapsed_sec": float(elapsed),
        "split_val_MAE": mae(y_blend_val, pred_val),
        "fullfit_typical_MAE": mae(ytest_typical, pred_test_typ_fullfit),
        "fullfit_full_MAE": mae(ytest_full, pred_test_full_fullfit),
    }

def build_prediction_matrices_ml(pool_frame: pd.DataFrame):
    rows = []
    for _, row in pool_frame.iterrows():
        try:
            out = fit_and_predict_tuned_ml_model(row)
            rows.append(out)
        except Exception as e:
            rows.append({"model": row["model"], "group": "ML", "status": "failed", "error": repr(e)})
            print(f"FAILED ML: {row['model']} -> {e}")
        cleanup_memory()
    return pd.DataFrame(rows)


In [ ]:

# =========================
# Ensemble utilities
# =========================
def clip_pred(pred):
    return np.clip(np.asarray(pred, dtype=float), 0, None)

def mae(y, p):
    return float(mean_absolute_error(y, clip_pred(p)))

def rmse(y, p):
    return float(np.sqrt(mean_squared_error(y, clip_pred(p))))

def mdae(y, p):
    return float(median_absolute_error(y, clip_pred(p)))

def aggregate_predictions(pred_matrix: np.ndarray, method: str, trim_frac: float = 0.1):
    pred_matrix = np.asarray(pred_matrix, dtype=float)
    if pred_matrix.ndim == 1:
        return clip_pred(pred_matrix)
    if method == "mean":
        out = pred_matrix.mean(axis=1)
    elif method == "median":
        out = np.median(pred_matrix, axis=1)
    elif method == "trimmed_mean":
        m = pred_matrix.shape[1]
        k = int(np.floor(m * trim_frac))
        if k <= 0 or 2 * k >= m:
            out = pred_matrix.mean(axis=1)
        else:
            sorted_mat = np.sort(pred_matrix, axis=1)
            out = sorted_mat[:, k:m-k].mean(axis=1)
    else:
        raise KeyError(method)
    return clip_pred(out)

def normalize_weights(w):
    w = np.asarray(w, dtype=float)
    w = np.where(np.isfinite(w), w, 0.0)
    w = np.clip(w, 0, None)
    s = w.sum()
    if s <= 0:
        return np.ones_like(w) / len(w)
    return w / s

def weights_from_errors(pred_fit: pd.DataFrame, y_fit, scheme: str, temperature: float = 1.0, power: float = 1.0):
    errs = np.array([mae(y_fit, pred_fit[c].values) for c in pred_fit.columns], dtype=float)
    if scheme == "equal":
        w = np.ones(len(errs), dtype=float)
    elif scheme == "inverse_mae":
        w = 1.0 / np.maximum(errs, 1e-9)
    elif scheme == "inverse_mae_sq":
        w = 1.0 / np.maximum(errs, 1e-9) ** 2
    elif scheme == "softmax_mae":
        z = -errs / max(float(temperature), 1e-6)
        z = z - z.max()
        w = np.exp(z)
    elif scheme == "rank_power":
        order = np.argsort(errs)
        ranks = np.empty_like(order)
        ranks[order] = np.arange(1, len(errs) + 1)
        w = 1.0 / np.power(ranks, power)
    else:
        raise KeyError(scheme)
    return normalize_weights(w)

def fit_weighted_blender(pred_fit: pd.DataFrame, y_fit, scheme: str, **kwargs):
    X = pred_fit.values.astype(float)
    if scheme in {"equal", "inverse_mae", "inverse_mae_sq", "softmax_mae", "rank_power"}:
        w = weights_from_errors(pred_fit, y_fit, scheme, **kwargs)
        return {"kind": "weights", "weights": w}
    elif scheme == "nnls":
        w, _ = nnls(X, np.asarray(y_fit, dtype=float))
        w = normalize_weights(w)
        return {"kind": "weights", "weights": w}
    elif scheme == "simplex_mae":
        n = X.shape[1]
        x0 = np.ones(n, dtype=float) / n
        bounds = [(0.0, 1.0)] * n
        cons = {"type": "eq", "fun": lambda w: np.sum(w) - 1.0}
        def objective(w):
            return mae(y_fit, X @ w)
        res = minimize(objective, x0=x0, method="SLSQP", bounds=bounds, constraints=[cons], options={"maxiter": 500, "ftol": 1e-8})
        w = normalize_weights(res.x if res.success else x0)
        return {"kind": "weights", "weights": w, "optimizer_success": bool(res.success)}
    else:
        raise KeyError(scheme)

def predict_weighted_blender(fitted, pred_df: pd.DataFrame):
    w = np.asarray(fitted["weights"], dtype=float)
    return clip_pred(pred_df.values @ w)

def build_stacker(stacker_name: str):
    if stacker_name == "linear":
        return LinearRegression()
    elif stacker_name == "positive_linear":
        return LinearRegression(positive=True)
    elif stacker_name == "ridge":
        return RidgeCV(alphas=np.logspace(-4, 4, 25))
    elif stacker_name == "lasso":
        return LassoCV(alphas=np.logspace(-4, 1, 20), cv=5, random_state=SEED, max_iter=20000)
    elif stacker_name == "elastic":
        return ElasticNetCV(alphas=np.logspace(-4, 1, 20), l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.98], cv=5, random_state=SEED, max_iter=20000)
    elif stacker_name == "bayes":
        return BayesianRidge()
    elif stacker_name == "huber":
        return HuberRegressor(max_iter=1000, epsilon=1.35)
    elif stacker_name == "quantile":
        return QuantileRegressor(quantile=0.5, alpha=1e-4, solver="highs")
    elif stacker_name == "gbr":
        return GradientBoostingRegressor(loss="absolute_error", random_state=SEED, n_estimators=300, learning_rate=0.05, max_depth=2, min_samples_leaf=5, min_samples_split=4, subsample=0.9)
    elif stacker_name == "rf":
        return RandomForestRegressor(n_estimators=500, random_state=SEED, min_samples_leaf=2, n_jobs=-1)
    elif stacker_name == "etr":
        return ExtraTreesRegressor(n_estimators=500, random_state=SEED, min_samples_leaf=2, n_jobs=-1)
    elif stacker_name == "xgb":
        if not HAS_XGB_FOR_STACKING:
            raise RuntimeError("xgboost недоступен")
        return XGBRegressor(
            objective="reg:absoluteerror",
            eval_metric="mae",
            n_estimators=500,
            max_depth=2,
            learning_rate=0.03,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_alpha=0.0,
            reg_lambda=1.0,
            random_state=SEED,
            tree_method="hist",
            verbosity=0,
        )
    else:
        raise KeyError(stacker_name)

def fit_stacker(pred_fit: pd.DataFrame, y_fit, stacker_name: str):
    model = build_stacker(stacker_name)
    model.fit(pred_fit.values.astype(float), np.asarray(y_fit, dtype=float))
    return {"kind": "stacker", "model": model, "stacker_name": stacker_name}

def predict_stacker(fitted, pred_df: pd.DataFrame):
    pred = fitted["model"].predict(pred_df.values.astype(float))
    return clip_pred(pred)

def spec_tag(spec: dict) -> str:
    fam = spec["family"]
    models = ",".join(spec["models"])
    extra = spec.get("name", fam)
    return f"{fam}::{extra}::{models}"

def evaluate_spec(spec: dict, pred_fit_df: pd.DataFrame, y_fit, pred_sel_df: pd.DataFrame, y_sel):
    models = spec["models"]
    X_fit = pred_fit_df[models]
    X_sel = pred_sel_df[models]

    fitted = None
    if spec["family"] == "single":
        pred_sel = X_sel.iloc[:, 0].values
    elif spec["family"] == "aggregate":
        pred_sel = aggregate_predictions(X_sel.values, method=spec["agg_method"], trim_frac=spec.get("trim_frac", 0.1))
    elif spec["family"] == "weighted":
        fitted = fit_weighted_blender(X_fit, y_fit, spec["weight_scheme"], **spec.get("weight_kwargs", {}))
        pred_sel = predict_weighted_blender(fitted, X_sel)
    elif spec["family"] == "pair_grid":
        pair_cols = X_fit.columns.tolist()
        best_w, best_fit_mae = 0.5, float("inf")
        fit_a = X_fit[pair_cols[0]].values
        fit_b = X_fit[pair_cols[1]].values
        for w in PAIR_WEIGHT_GRID:
            p = w * fit_a + (1.0 - w) * fit_b
            cur = mae(y_fit, p)
            if cur < best_fit_mae:
                best_fit_mae = cur
                best_w = float(w)
        fitted = {"kind": "pair_grid", "weight_first": best_w}
        pred_sel = best_w * X_sel[pair_cols[0]].values + (1.0 - best_w) * X_sel[pair_cols[1]].values
    elif spec["family"] == "stacker":
        fitted = fit_stacker(X_fit, y_fit, spec["stacker_name"])
        pred_sel = predict_stacker(fitted, X_sel)
    else:
        raise KeyError(spec["family"])

    row = {
        "tag": spec_tag(spec),
        "family": spec["family"],
        "name": spec.get("name", spec["family"]),
        "n_models": len(models),
        "models": models,
        "selection_MAE": mae(y_sel, pred_sel),
        "selection_RMSE": rmse(y_sel, pred_sel),
        "selection_MdAE": mdae(y_sel, pred_sel),
        "fitted_obj": fitted,
        "spec": spec,
    }
    return row

def refit_and_eval_spec(spec: dict, pred_val_df: pd.DataFrame, y_val, pred_test_typ_df: pd.DataFrame, y_test_typ, pred_test_full_df: pd.DataFrame, y_test_full):
    models = spec["models"]
    X_val = pred_val_df[models]
    X_test_typ = pred_test_typ_df[models]
    X_test_full = pred_test_full_df[models]

    if spec["family"] == "single":
        pred_typ = X_test_typ.iloc[:, 0].values
        pred_full = X_test_full.iloc[:, 0].values
        fitted = None
    elif spec["family"] == "aggregate":
        pred_typ = aggregate_predictions(X_test_typ.values, method=spec["agg_method"], trim_frac=spec.get("trim_frac", 0.1))
        pred_full = aggregate_predictions(X_test_full.values, method=spec["agg_method"], trim_frac=spec.get("trim_frac", 0.1))
        fitted = None
    elif spec["family"] == "weighted":
        fitted = fit_weighted_blender(X_val, y_val, spec["weight_scheme"], **spec.get("weight_kwargs", {}))
        pred_typ = predict_weighted_blender(fitted, X_test_typ)
        pred_full = predict_weighted_blender(fitted, X_test_full)
    elif spec["family"] == "pair_grid":
        pair_cols = X_val.columns.tolist()
        best_w, best_val_mae = 0.5, float("inf")
        for w in PAIR_WEIGHT_GRID:
            p = w * X_val[pair_cols[0]].values + (1.0 - w) * X_val[pair_cols[1]].values
            cur = mae(y_val, p)
            if cur < best_val_mae:
                best_val_mae = cur
                best_w = float(w)
        fitted = {"kind": "pair_grid", "weight_first": best_w}
        pred_typ = best_w * X_test_typ[pair_cols[0]].values + (1.0 - best_w) * X_test_typ[pair_cols[1]].values
        pred_full = best_w * X_test_full[pair_cols[0]].values + (1.0 - best_w) * X_test_full[pair_cols[1]].values
    elif spec["family"] == "stacker":
        fitted = fit_stacker(X_val, y_val, spec["stacker_name"])
        pred_typ = predict_stacker(fitted, X_test_typ)
        pred_full = predict_stacker(fitted, X_test_full)
    else:
        raise KeyError(spec["family"])

    typ_metrics = calc_reg_metrics(y_test_typ, pred_typ)
    full_metrics = calc_reg_metrics(y_test_full, pred_full)
    out = {
        "tag": spec_tag(spec),
        "family": spec["family"],
        "name": spec.get("name", spec["family"]),
        "n_models": len(models),
        "models": models,
        "MAE_typical": typ_metrics["MAE"],
        "RMSE_typical": typ_metrics["RMSE"],
        "MdAE_typical": typ_metrics["MdAE"],
        "MAE_full": full_metrics["MAE"],
        "RMSE_full": full_metrics["RMSE"],
        "MdAE_full": full_metrics["MdAE"],
        "spec": spec,
        "fitted_obj": fitted,
    }
    if fitted and fitted.get("kind") == "weights":
        out["weights"] = fitted["weights"].tolist()
    if fitted and fitted.get("kind") == "pair_grid":
        out["weight_first"] = float(fitted["weight_first"])
    if fitted and fitted.get("kind") == "stacker":
        model = fitted["model"]
        if hasattr(model, "coef_"):
            out["weights_like"] = np.ravel(model.coef_).tolist()
        elif hasattr(model, "feature_importances_"):
            out["weights_like"] = np.ravel(model.feature_importances_).tolist()
    return out

def top_models_by_fit_mae(pred_fit_df: pd.DataFrame, y_fit):
    rows = []
    for c in pred_fit_df.columns:
        rows.append((c, mae(y_fit, pred_fit_df[c].values)))
    rows.sort(key=lambda x: x[1])
    return [m for m, _ in rows], pd.DataFrame(rows, columns=["model", "blend_fit_MAE"])

def greedy_forward_mean(pred_fit_df: pd.DataFrame, y_fit, ranked_models):
    selected = [ranked_models[0]]
    best_fit = mae(y_fit, pred_fit_df[selected].mean(axis=1).values)
    history = [{"step": 1, "models": selected.copy(), "fit_MAE": best_fit}]
    remaining = ranked_models[1:].copy()
    while remaining:
        best_candidate = None
        best_candidate_mae = best_fit
        for cand in remaining:
            trial_models = selected + [cand]
            trial_mae = mae(y_fit, pred_fit_df[trial_models].mean(axis=1).values)
            if trial_mae < best_candidate_mae - 1e-9:
                best_candidate = cand
                best_candidate_mae = trial_mae
        if best_candidate is None:
            break
        selected.append(best_candidate)
        remaining.remove(best_candidate)
        best_fit = best_candidate_mae
        history.append({"step": len(selected), "models": selected.copy(), "fit_MAE": best_fit})
    return history

def greedy_forward_simplex(pred_fit_df: pd.DataFrame, y_fit, ranked_models):
    selected = [ranked_models[0]]
    fitted = fit_weighted_blender(pred_fit_df[selected], y_fit, "simplex_mae")
    best_fit = mae(y_fit, predict_weighted_blender(fitted, pred_fit_df[selected]))
    history = [{"step": 1, "models": selected.copy(), "fit_MAE": best_fit}]
    remaining = ranked_models[1:].copy()
    while remaining:
        best_candidate = None
        best_candidate_mae = best_fit
        for cand in remaining:
            trial_models = selected + [cand]
            fitted_trial = fit_weighted_blender(pred_fit_df[trial_models], y_fit, "simplex_mae")
            trial_mae = mae(y_fit, predict_weighted_blender(fitted_trial, pred_fit_df[trial_models]))
            if trial_mae < best_candidate_mae - 1e-9:
                best_candidate = cand
                best_candidate_mae = trial_mae
        if best_candidate is None:
            break
        selected.append(best_candidate)
        remaining.remove(best_candidate)
        best_fit = best_candidate_mae
        history.append({"step": len(selected), "models": selected.copy(), "fit_MAE": best_fit})
    return history


In [ ]:
import os, shutil
from pathlib import Path
from huggingface_hub import hf_hub_download

TABPFN_CACHE = Path("/content/drive/MyDrive/diploma_dl_runs/tabpfn_model_cache")
TABPFN_CACHE.mkdir(parents=True, exist_ok=True)

TARGET = TABPFN_CACHE / "tabpfn-v2.5-regressor-v2.5_default.ckpt"

hf_token = (
    os.environ.get("HF_TOKEN")
    or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    or os.environ.get("HUGGINGFACEHUB_API_TOKEN")
    or os.environ.get("HUGGINGFACE_TOKEN")
)

if not hf_token:
    try:
        from google.colab import userdata
        hf_token = (
            userdata.get("HF_TOKEN")
            or userdata.get("HUGGING_FACE_HUB_TOKEN")
            or userdata.get("HUGGINGFACEHUB_API_TOKEN")
            or userdata.get("HUGGINGFACE_TOKEN")
        )
    except Exception:
        pass

print("HF token exists:", bool(hf_token))

downloaded = hf_hub_download(
    repo_id="Prior-Labs/tabpfn_2_5",
    filename="tabpfn-v2.5-regressor-v2.5_default.ckpt",
    token=hf_token,
    force_download=True,
)

print("Downloaded to:", downloaded)

shutil.copyfile(downloaded, TARGET)

os.environ["TABPFN_MODEL_CACHE_DIR"] = str(TABPFN_CACHE)
print("Target exists:", TARGET.exists(), "size:", TARGET.stat().st_size if TARGET.exists() else None)
print("TABPFN_MODEL_CACHE_DIR =", os.environ["TABPFN_MODEL_CACHE_DIR"])

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso, ElasticNet, Ridge, BayesianRidge, HuberRegressor
# =========================
# 1) Rebuild tuned DL + tuned ML models -> cache predictions
# =========================
rebuild_log_dl_df = build_prediction_matrices(POOL_DF)
rebuild_log_ml_df = build_prediction_matrices_ml(ML_POOL_DF)

rebuild_log_dl_df["group"] = rebuild_log_dl_df.get("group", "DL")
rebuild_log_ml_df["group"] = rebuild_log_ml_df.get("group", "ML")
rebuild_log_df = pd.concat([rebuild_log_dl_df, rebuild_log_ml_df], ignore_index=True, sort=False)

display(rebuild_log_df)
rebuild_log_df.to_csv(RUN_DIR / "rebuild_log.csv", index=False)
rebuild_log_df.to_csv(ART_DIR / "rebuild_log_latest.csv", index=False)
rebuild_log_dl_df.to_csv(RUN_DIR / "rebuild_log_dl.csv", index=False)
rebuild_log_ml_df.to_csv(RUN_DIR / "rebuild_log_ml.csv", index=False)

# Collect prediction matrices from both groups
combined_selected_models = SELECTED_MODELS + ML_SELECTED_MODELS
model_group_map = {m: "DL" for m in SELECTED_MODELS}
model_group_map.update({m: "ML" for m in ML_SELECTED_MODELS})

val_pred_df = load_prediction_frame("split_val", combined_selected_models)
test_typ_split_pred_df = load_prediction_frame("split_test_typical", combined_selected_models)
test_full_split_pred_df = load_prediction_frame("split_test_full", combined_selected_models)
test_typ_fullfit_pred_df = load_prediction_frame("fullfit_test_typical", combined_selected_models)
test_full_fullfit_pred_df = load_prediction_frame("fullfit_test_full", combined_selected_models)

available_models = [m for m in combined_selected_models if m in val_pred_df.columns and m in test_typ_fullfit_pred_df.columns and m in test_full_fullfit_pred_df.columns]
val_pred_df = val_pred_df[available_models].copy()
test_typ_split_pred_df = test_typ_split_pred_df[available_models].copy()
test_full_split_pred_df = test_full_split_pred_df[available_models].copy()
test_typ_fullfit_pred_df = test_typ_fullfit_pred_df[available_models].copy()
test_full_fullfit_pred_df = test_full_fullfit_pred_df[available_models].copy()

print(f"Available rebuilt models: {len(available_models)}")
print(available_models)

if len(available_models) < 2:
    raise RuntimeError("Для ансамблей нужно хотя бы 2 успешно восстановленные модели.")

# save prediction matrices
val_pred_df.to_csv(RUN_DIR / "base_pred_split_val.csv", index=False)
val_pred_df.to_csv(ART_DIR / "base_pred_split_val_latest.csv", index=False)
test_typ_fullfit_pred_df.to_csv(RUN_DIR / "base_pred_fullfit_test_typical.csv", index=False)
test_typ_fullfit_pred_df.to_csv(ART_DIR / "base_pred_fullfit_test_typical_latest.csv", index=False)
test_full_fullfit_pred_df.to_csv(RUN_DIR / "base_pred_fullfit_test_full.csv", index=False)
test_full_fullfit_pred_df.to_csv(ART_DIR / "base_pred_fullfit_test_full_latest.csv", index=False)

# inner validation targets for ensemble search
y_val = np.asarray(y_dl_va, dtype=float)
y_test_typ = np.asarray(ytest_typical, dtype=float)
y_test_full = np.asarray(ytest_full, dtype=float)

base_leaderboard = get_base_leaderboard_from_predictions(val_pred_df)
base_leaderboard["group"] = base_leaderboard["model"].map(model_group_map)
display(base_leaderboard.head(20))
base_leaderboard.to_csv(RUN_DIR / "rebuilt_base_leaderboard.csv", index=False)
base_leaderboard.to_csv(ART_DIR / "rebuilt_base_leaderboard_latest.csv", index=False)

# inner-val -> blend_fit / blend_select
blend_cut = int(len(y_val) * BLEND_FIT_FRAC)
if blend_cut <= 0 or blend_cut >= len(y_val):
    raise ValueError("Некорректный BLEND_FIT_FRAC.")
pred_fit_df = val_pred_df.iloc[:blend_cut].copy()
pred_sel_df = val_pred_df.iloc[blend_cut:].copy()
y_fit = y_val[:blend_cut].copy()
y_sel = y_val[blend_cut:].copy()

ranked_models, fit_rank_df = top_models_by_fit_mae(pred_fit_df, y_fit)
fit_rank_df["group"] = fit_rank_df["model"].map(model_group_map)
print("Top models by blend-fit MAE:")
display(fit_rank_df.head(20))
fit_rank_df.to_csv(RUN_DIR / "blend_fit_model_ranking.csv", index=False)
fit_rank_df.to_csv(ART_DIR / "blend_fit_model_ranking_latest.csv", index=False)

# =========================
# 2) Generate ensemble specs
# =========================
specs = []

# singles
for m in ranked_models:
    specs.append({"family": "single", "name": "single", "models": [m]})

# global aggregations on all models
all_models = ranked_models.copy()
specs.extend([
    {"family": "aggregate", "name": "all_mean", "models": all_models, "agg_method": "mean"},
    {"family": "aggregate", "name": "all_median", "models": all_models, "agg_method": "median"},
    {"family": "aggregate", "name": "all_trim10", "models": all_models, "agg_method": "trimmed_mean", "trim_frac": 0.10},
    {"family": "aggregate", "name": "all_trim20", "models": all_models, "agg_method": "trimmed_mean", "trim_frac": 0.20},
])

# group-specific aggregates
ml_models_ranked = [m for m in ranked_models if model_group_map.get(m) == "ML"]
dl_models_ranked = [m for m in ranked_models if model_group_map.get(m) == "DL"]
if len(ml_models_ranked) >= 2:
    specs.extend([
        {"family": "aggregate", "name": "ml_only_mean", "models": ml_models_ranked, "agg_method": "mean"},
        {"family": "aggregate", "name": "ml_only_median", "models": ml_models_ranked, "agg_method": "median"},
    ])
if len(dl_models_ranked) >= 2:
    specs.extend([
        {"family": "aggregate", "name": "dl_only_mean", "models": dl_models_ranked, "agg_method": "mean"},
        {"family": "aggregate", "name": "dl_only_median", "models": dl_models_ranked, "agg_method": "median"},
    ])

# top-k prefix ensembles
if RUN_TOPK_PREFIX_ENSEMBLES:
    for k in range(2, len(ranked_models) + 1):
        topk = ranked_models[:k]
        specs.append({"family": "aggregate", "name": f"top{k}_mean", "models": topk, "agg_method": "mean"})
        specs.append({"family": "aggregate", "name": f"top{k}_median", "models": topk, "agg_method": "median"})
        if k >= 5:
            specs.append({"family": "aggregate", "name": f"top{k}_trim10", "models": topk, "agg_method": "trimmed_mean", "trim_frac": 0.10})
        if k >= 8:
            specs.append({"family": "aggregate", "name": f"top{k}_trim20", "models": topk, "agg_method": "trimmed_mean", "trim_frac": 0.20})

        specs.append({"family": "weighted", "name": f"top{k}_inverse_mae", "models": topk, "weight_scheme": "inverse_mae"})
        specs.append({"family": "weighted", "name": f"top{k}_inverse_mae_sq", "models": topk, "weight_scheme": "inverse_mae_sq"})
        for temp in [0.25, 0.5, 1.0, 2.0, 4.0]:
            specs.append({"family": "weighted", "name": f"top{k}_softmax_t{temp}", "models": topk, "weight_scheme": "softmax_mae", "weight_kwargs": {"temperature": temp}})
        for power in [0.5, 1.0, 2.0]:
            specs.append({"family": "weighted", "name": f"top{k}_rankpow_{power}", "models": topk, "weight_scheme": "rank_power", "weight_kwargs": {"power": power}})
        specs.append({"family": "weighted", "name": f"top{k}_nnls", "models": topk, "weight_scheme": "nnls"})
        specs.append({"family": "weighted", "name": f"top{k}_simplex_mae", "models": topk, "weight_scheme": "simplex_mae"})

# all pairs
if RUN_ALL_PAIRS:
    for a, b in combinations(ranked_models, 2):
        specs.append({"family": "aggregate", "name": "pair_mean", "models": [a, b], "agg_method": "mean"})
        specs.append({"family": "pair_grid", "name": "pair_grid", "models": [a, b]})

# all triples
if RUN_ALL_TRIPLES:
    for trio in combinations(ranked_models, 3):
        specs.append({"family": "aggregate", "name": "triple_mean", "models": list(trio), "agg_method": "mean"})

# exhaustive subsets on top-N
if RUN_EXHAUSTIVE_TOPN_SUBSETS:
    topn_models = ranked_models[:min(EXHAUSTIVE_TOP_N, len(ranked_models))]
    for r in range(2, len(topn_models) + 1):
        for combo in combinations(topn_models, r):
            specs.append({"family": "aggregate", "name": f"topN{len(topn_models)}_subset_mean", "models": list(combo), "agg_method": "mean"})

# greedy forward searches
if RUN_GREEDY_SEARCH:
    mean_path = greedy_forward_mean(pred_fit_df[ranked_models], y_fit, ranked_models)
    simplex_path = greedy_forward_simplex(pred_fit_df[ranked_models], y_fit, ranked_models)
    for step in mean_path:
        specs.append({"family": "aggregate", "name": f"greedy_mean_step{step['step']}", "models": step["models"], "agg_method": "mean"})
    for step in simplex_path:
        specs.append({"family": "weighted", "name": f"greedy_simplex_step{step['step']}", "models": step["models"], "weight_scheme": "simplex_mae"})

# stackers
if RUN_STACKERS:
    pool_variants = {
        "top5": ranked_models[:min(5, len(ranked_models))],
        "top10": ranked_models[:min(10, len(ranked_models))],
        "all": ranked_models,
    }
    stacker_names = ["linear", "positive_linear", "ridge", "lasso", "elastic", "bayes", "huber", "quantile", "gbr", "rf", "etr"]
    if HAS_XGB_FOR_STACKING:
        stacker_names.append("xgb")
    for pool_name, pool_models in pool_variants.items():
        if len(pool_models) >= 2:
            for stk in stacker_names:
                specs.append({"family": "stacker", "name": f"{stk}_{pool_name}", "models": pool_models, "stacker_name": stk})

print(f"Total ensemble specs to evaluate on blend_select: {len(specs)}")

# =========================
# 3) Evaluate specs on blend_select
# =========================
search_rows = []
for idx, spec in enumerate(specs, start=1):
    if idx % 250 == 0:
        print(f"Evaluated {idx}/{len(specs)} specs ...")
    row = evaluate_spec(spec, pred_fit_df, y_fit, pred_sel_df, y_sel)
    row["n_ml_models"] = int(sum(model_group_map.get(m) == "ML" for m in row["models"]))
    row["n_dl_models"] = int(sum(model_group_map.get(m) == "DL" for m in row["models"]))
    search_rows.append(row)

search_df = pd.DataFrame([
    {
        "tag": r["tag"],
        "family": r["family"],
        "name": r["name"],
        "n_models": r["n_models"],
        "n_ml_models": r["n_ml_models"],
        "n_dl_models": r["n_dl_models"],
        "models": json.dumps(r["models"], ensure_ascii=False),
        "selection_MAE": r["selection_MAE"],
        "selection_RMSE": r["selection_RMSE"],
        "selection_MdAE": r["selection_MdAE"],
    }
    for r in search_rows
]).sort_values(["selection_MAE", "selection_RMSE", "n_models"], kind="stable").reset_index(drop=True)

display(search_df.head(30))
search_df.to_csv(RUN_DIR / "ensemble_search_leaderboard.csv", index=False)
search_df.to_csv(ART_DIR / "ensemble_search_leaderboard_latest.csv", index=False)

search_rows_sorted = sorted(search_rows, key=lambda r: (r["selection_MAE"], r["selection_RMSE"], r["n_models"]))
top_refit_rows = search_rows_sorted[:min(REFIT_TOP_ENSEMBLES, len(search_rows_sorted))]
print(f"Top specs selected for final holdout refit/eval: {len(top_refit_rows)}")


In [ ]:

# =========================
# 4) Refit top ensembles on full inner-val and evaluate on holdout
# =========================
final_rows = []
for idx, row in enumerate(top_refit_rows, start=1):
    if idx % 25 == 0:
        print(f"Final holdout refit {idx}/{len(top_refit_rows)} ...")
    spec = row["spec"]
    out = refit_and_eval_spec(
        spec,
        pred_val_df=val_pred_df,
        y_val=y_val,
        pred_test_typ_df=test_typ_fullfit_pred_df,
        y_test_typ=y_test_typ,
        pred_test_full_df=test_full_fullfit_pred_df,
        y_test_full=y_test_full,
    )
    out["selection_MAE"] = row["selection_MAE"]
    out["selection_RMSE"] = row["selection_RMSE"]
    out["selection_MdAE"] = row["selection_MdAE"]
    out["n_ml_models"] = int(sum(model_group_map.get(m) == "ML" for m in out["models"]))
    out["n_dl_models"] = int(sum(model_group_map.get(m) == "DL" for m in out["models"]))
    final_rows.append(out)

final_df = pd.DataFrame([
    {
        "tag": r["tag"],
        "family": r["family"],
        "name": r["name"],
        "n_models": r["n_models"],
        "n_ml_models": r["n_ml_models"],
        "n_dl_models": r["n_dl_models"],
        "models": json.dumps(r["models"], ensure_ascii=False),
        "model_groups": json.dumps([model_group_map.get(m, "?") for m in r["models"]], ensure_ascii=False),
        "selection_MAE": r["selection_MAE"],
        "selection_RMSE": r["selection_RMSE"],
        "selection_MdAE": r["selection_MdAE"],
        "MAE_typical": r["MAE_typical"],
        "RMSE_typical": r["RMSE_typical"],
        "MdAE_typical": r["MdAE_typical"],
        "MAE_full": r["MAE_full"],
        "RMSE_full": r["RMSE_full"],
        "MdAE_full": r["MdAE_full"],
        "weights": json.dumps(r.get("weights", []), ensure_ascii=False),
        "weights_like": json.dumps(r.get("weights_like", []), ensure_ascii=False),
        "weight_first": r.get("weight_first", np.nan),
    }
    for r in final_rows
]).sort_values(["selection_MAE", "MAE_typical", "MAE_full"], kind="stable").reset_index(drop=True)

display(final_df.head(30))
final_df.to_csv(RUN_DIR / "ensemble_holdout_leaderboard.csv", index=False)
final_df.to_csv(ART_DIR / "ensemble_holdout_leaderboard_latest.csv", index=False)

best_ensemble_row = final_rows[0]
best_single_model = min(available_models, key=lambda m: mae(y_val, val_pred_df[m].values))
best_single_group = model_group_map.get(best_single_model, "unknown")

single_typ = calc_reg_metrics(y_test_typ, test_typ_fullfit_pred_df[best_single_model].values)
single_full = calc_reg_metrics(y_test_full, test_full_fullfit_pred_df[best_single_model].values)

comparison_payload = {
    "selection_metric": "blend_select_MAE",
    "blend_fit_frac": BLEND_FIT_FRAC,
    "available_models": available_models,
    "available_models_ml": [m for m in available_models if model_group_map.get(m) == "ML"],
    "available_models_dl": [m for m in available_models if model_group_map.get(m) == "DL"],
    "best_single_model_by_val": best_single_model,
    "best_single_model_group": best_single_group,
    "best_single_model_metrics": {
        "MAE_typical": single_typ["MAE"],
        "RMSE_typical": single_typ["RMSE"],
        "MdAE_typical": single_typ["MdAE"],
        "MAE_full": single_full["MAE"],
        "RMSE_full": single_full["RMSE"],
        "MdAE_full": single_full["MdAE"],
    },
    "best_ensemble": {
        "tag": best_ensemble_row["tag"],
        "family": best_ensemble_row["family"],
        "name": best_ensemble_row["name"],
        "models": best_ensemble_row["models"],
        "n_ml_models": best_ensemble_row["n_ml_models"],
        "n_dl_models": best_ensemble_row["n_dl_models"],
        "selection_MAE": best_ensemble_row["selection_MAE"],
        "MAE_typical": best_ensemble_row["MAE_typical"],
        "RMSE_typical": best_ensemble_row["RMSE_typical"],
        "MdAE_typical": best_ensemble_row["MdAE_typical"],
        "MAE_full": best_ensemble_row["MAE_full"],
        "RMSE_full": best_ensemble_row["RMSE_full"],
        "MdAE_full": best_ensemble_row["MdAE_full"],
        "weights": best_ensemble_row.get("weights", []),
        "weights_like": best_ensemble_row.get("weights_like", []),
        "weight_first": best_ensemble_row.get("weight_first", None),
    },
}

display(pd.DataFrame([
    {"item": "best_single_model_by_val", "value": best_single_model},
    {"item": "best_single_model_group", "value": best_single_group},
    {"item": "best_ensemble_tag", "value": best_ensemble_row["tag"]},
    {"item": "best_ensemble_selection_MAE", "value": best_ensemble_row["selection_MAE"]},
    {"item": "best_ensemble_MAE_typical", "value": best_ensemble_row["MAE_typical"]},
    {"item": "best_ensemble_MAE_full", "value": best_ensemble_row["MAE_full"]},
]))

with open(RUN_DIR / "best_ensemble_summary.json", "w", encoding="utf-8") as f:
    json.dump(comparison_payload, f, ensure_ascii=False, indent=2)
with open(ART_DIR / "best_ensemble_summary_latest.json", "w", encoding="utf-8") as f:
    json.dump(comparison_payload, f, ensure_ascii=False, indent=2)

best_models = best_ensemble_row["models"]
best_weights = best_ensemble_row.get("weights") or best_ensemble_row.get("weights_like") or []
if len(best_weights) == len(best_models) and len(best_weights) > 0:
    wdf = pd.DataFrame({"model": best_models, "group": [model_group_map.get(m, "?") for m in best_models], "weight": best_weights})
    wdf.to_csv(RUN_DIR / "best_ensemble_weights.csv", index=False)
    wdf.to_csv(ART_DIR / "best_ensemble_weights_latest.csv", index=False)

run_manifest = {
    "run_name": RUN_NAME,
    "data_path": str(DATA_PATH),
    "dl_tuned_summary_path": str(TUNED_SUMMARY_PATH),
    "ml_tuned_summary_path": str(ML_TUNED_SUMMARY_PATH),
    "artifacts_root": str(ART_DIR),
    "run_dir": str(RUN_DIR),
    "duration_cap": DURATION_CAP,
    "train_full_rows": int(len(train_full)),
    "train_typical_rows": int(len(train_typical)),
    "test_typical_rows": int(len(test_typical)),
    "test_full_rows": int(len(test_full)),
    "selected_models": available_models,
    "selected_models_ml": [m for m in available_models if model_group_map.get(m) == "ML"],
    "selected_models_dl": [m for m in available_models if model_group_map.get(m) == "DL"],
    "blend_fit_frac": BLEND_FIT_FRAC,
    "search_specs_total": int(len(specs)),
    "search_specs_refit": int(len(top_refit_rows)),
    "tabpfn_hf_login_status": globals().get("TABPFN_HF_LOGIN_STATUS", "unknown"),
    "best_single_model_by_val": best_single_model,
    "best_single_model_group": best_single_group,
    "best_ensemble_tag": best_ensemble_row["tag"],
}
with open(RUN_DIR / "run_manifest.json", "w", encoding="utf-8") as f:
    json.dump(run_manifest, f, ensure_ascii=False, indent=2)
with open(ART_DIR / "run_manifest_latest.json", "w", encoding="utf-8") as f:
    json.dump(run_manifest, f, ensure_ascii=False, indent=2)

print("ML+DL ensemble experiment finished. Main artifacts:")
print(RUN_DIR / "ensemble_search_leaderboard.csv")
print(RUN_DIR / "ensemble_holdout_leaderboard.csv")
print(RUN_DIR / "best_ensemble_summary.json")


In [ ]:
top_plot = final_df.head(15).copy()

plt.figure(figsize=(12, 6))

colors = top_plot.apply(
    lambda row: (
        "tab:blue" if row["n_ml_models"] == 0
        else ("tab:orange" if row["n_ml_models"] == row["n_models"] else "tab:green")
    ),
    axis=1
)

plt.barh(top_plot["tag"][::-1], top_plot["MAE_typical"][::-1], color=list(colors[::-1]))
plt.xlabel("MAE_typical")
plt.title("Top ensembles by holdout_typical_MAE")
plt.tight_layout()
plt.show()

display(final_df.head(20))

In [ ]:

# =========================
# 5) Quick plots / diagnostics
# =========================
top_plot = final_df.head(15).copy()
plt.figure(figsize=(12, 6))
colors = top_plot["n_ml_models"].apply(lambda n: "tab:blue" if n == 0 else ("tab:orange" if n == top_plot["n_models"] else "tab:green"))
plt.barh(top_plot["tag"][::-1], top_plot["MAE_typical"][::-1], color=list(colors[::-1]))
plt.xlabel("MAE_typical")
plt.ylabel("Ensemble")
plt.title("Top-15 ML+DL ensembles by selection_MAE -> holdout typical")
plt.tight_layout()
plt.show()

top_corr_models = ranked_models[:min(12, len(ranked_models))]
corr = val_pred_df[top_corr_models].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Correlation of validation predictions (top-12 ML+DL base models)")
plt.tight_layout()
plt.show()

display(final_df.head(20))
